# Common Risk Factors in Cryptocurrency
## Nine long-short signals in two market-cap-weighted universes

We test whether coins ranked by size, price, recent returns or trading activity outperform coins at the opposite end of each ranking.
This is a comparison of nine separate portfolios, not a portfolio that combines all nine signals.
It adapts Liu, Tsyvinski and Wu's *Common Risk Factors in Cryptocurrency* to our available data.

**Reference:** Liu, Y., Tsyvinski, A., & Wu, X. (2022), *Common Risk Factors in Cryptocurrency*,
The Journal of Finance. [Published paper](https://doi.org/10.1111/jofi.13119) ·
[NBER working paper](https://www.nber.org/papers/w25882).


### What each strategy does

At each weekly rebalance, buy the highest-scoring quintile and short the lowest-scoring quintile.
Negative signs in the score mean that smaller observed values rank more highly.

| Strategy | Score used by this notebook | Long side | Short side |
|---|---|---|---|
| Size | Negative market capitalization | Smaller eligible coins | Larger eligible coins |
| Price | Negative closing price | Lower nominal prices | Higher nominal prices |
| Maximum price | Negative maximum close over 7 days | Lower recent maximum closes | Higher recent maximum closes |
| 1-week momentum | Close / close 7 days earlier - 1 | Stronger past-week returns | Weaker past-week returns |
| 2-week momentum | Close / close 14 days earlier - 1 | Stronger past-2-week returns | Weaker past-2-week returns |
| 3-week momentum | Close / close 21 days earlier - 1 | Stronger past-3-week returns | Weaker past-3-week returns |
| 4-week momentum | Close / close 28 days earlier - 1 | Stronger past-4-week returns | Weaker past-4-week returns |
| Volume level | Negative mean dollar turnover over 7 days | Lower trading activity | Higher trading activity |
| Volume volatility | Negative sample standard deviation of dollar turnover over 7 days | Less variable turnover | More variable turnover |

Low nominal price is not the same as cheap valuation. Volume volatility
measures variability of turnover, not price-return volatility. Maximum price uses daily closes as
an explicit proxy, not intraday highs. Rolling signals require complete windows and positive valid
volume; momentum includes both endpoint prices and all intervening daily observations.

### Trading process

1. Form signals through Sunday - Rank eligible coins using information available through the
   previous day; execute at Monday's close. Newly established positions first earn Tuesday's return.
2. Allocate 50% long and 50% short - Targets use after-fee NAV, for 100% gross and 0% net dollar
   exposure at rebalance. Positions drift thereafter. Fewer than 10 eligible coins or no useful ranking means cash.
3. Account for capital explicitly - Each strategy/period starts with 100,000 units of its account
   currency. Net accounts charge 20 bps per absolute traded notional, including event exits and final
   liquidation. Gross accounts omit that fee. NAV changes through position P&L and fees, not deposits.
4. Respect trading events - Apply verified announcement-driven exits and reset signals across
   suspensions. An unsupported held price or required exit stops the account and is reported.
5. Compare consistently - Final charts require completion in both backtest and validation periods; a negative return is not a failed execution. Bitcoin buy and hold provides a benchmark.

### Two main experiments

This notebook evaluates two universes, with all nine signals in each. These are two
experiment variants, not two individual signals. Both require valid prices/signals and a market cap
of at least $1 million, and both weight coins by market cap within each long and short basket.

| Experiment | Candidate universe | Cap source | Account currency |
|---|---|---|---|
| `cmc_30_cap` | Fixed 30 selected coins from the supplied CMC files | Daily CMC caps | USD |
| `binance_matched_cap` | Ordinary Binance USDT markets with usable historical CMC identity and qualifying caps | Exact preceding-Sunday CMC snapshots | USDT |

### Why use the broader Binance universe?

The companion strategy studies use a fixed basket of 30 coins. This notebook also tests a broader historical Binance universe to examine whether size, nominal price, momentum, and trading-activity effects persist among smaller coins. Does a wider universe reveal return patterns absent from the fixed basket? Does diversification reduce market exposure?

The downloader's catalogue contains 3,731 exchange-pair records, not 3,731 distinct eligible coins. This study uses the bundled USDT histories and admits a coin only when its identity, price history, signals, and historical market cap satisfy the declared rules. Short availability is not established for every coin, so these experiments study historical factor behavior rather than demonstrate an immediately executable short portfolio.

### Market-data repairs and checks

- Validate source files and identities - Check daily coverage, duplicate observations, CMC IDs,
  finite positive OHLC prices, valid volume and nonnegative caps; record source-file hashes.
- Repair demonstrable archive omissions - Supplement missing Binance observations with
  independently checksum-verified daily archives. Reject conflicting values; do not extrapolate beyond the observations recovered.
- Supply historical weekly snapshots - Load the expanded local CMC cache through September 13, 2026 with
  source-response hashes for the September 14 rebalance. Missing caps are never substituted with zero or a future observation.
- Separate reused identities - Keep old LUNA/Terra Classic separate from Terra 2.0. Do not stitch
  their prices, transfer holdings automatically, or manufacture airdrop proceeds.
- Review genuine extreme moves - Verify the May 2022 old-LUNA collapse against minute archives.
  Accept only those reviewed moves; retain the extreme-move guard elsewhere. LUNA and UST use
  observed pre-halt minute exits after a one-minute reaction delay, subject to a 1% minute-turnover limit.
- Complete missing event records - Add the announced AUD removal, TVK-to-VANRY and TOMO-to-VIC
  migrations, and STRAX redenomination. Use observed announcement-day closes before the halts;
  retain earlier price history and reset affected signals.
- Substantiate the UTK same-day exit - Verify the April 14, 2026 minute archive and apply
  the existing one-minute reaction delay, observed close, fee rate and 1% participation limit.
  Retain historical UTK positions; do not substitute swap proceeds or remove UTK from the universe.

These repairs use public historical observations and notices; archived announcement versions have
not all been independently reconstructed. Previously examined validation data is not an untouched prospective test. Borrow availability,
funding, borrow fees and margin liquidation are not modeled, so completion does not establish live
tradability or profitability.

Experiment dates: initial prices start January 1, 2021. Backtest returns run January 2, 2021 through
December 31, 2024; validation runs January 1, 2025 through September 17, 2026.

### Required local files
<!-- UPDATED: list all runtime data, repair evidence, manifests and checksum dependencies. -->

This notebook reads saved local files; it does not download prices, market caps or repair archives when it runs. Place `Coin_Prices/` beside this notebook; keep the remaining source and repair folders inside `data/`.

| Required location | Required contents and purpose |
|---|---|
| `OHLCV_Market_Cap_Data/` | `crypto_30_coin_daily_ohlcv.csv` and `crypto_30_coin_daily_market_cap_supply.csv`: prices, volume and daily caps for the fixed 30-coin universe. |
| `Coin_Prices/runs/20260918T174517572465Z/` | `config.json`, `instruments.csv`, `partition_manifest.csv`, and the 718 successful USDT `ohlcv/*.csv.gz` partitions selected by the manifest: Binance prices and volume. `SOURCE_RUN` selects this saved run. |
| `data/binance_cap_expansion/` | `manifest.json`, `identity_candidates.csv`, `weekly_observations.csv`, `additional_market_events.csv`, and every other file named in `manifest.json` under `files`: historical CMC caps, identity evidence and additional trading events. The notebook verifies all referenced files, including `fetch_audit.json`, pair/listing JSONs, `detail_*.json` and `announcement_*.json`. |
| `data/verified_market_events.csv` | The verified trading-event schedule used to block orders, close affected holdings and reset signals. |
| `data/backtest_repairs/` | `archive_manifest.json`, the 19 daily `NBTUSDT-1d-2022-09-DD.zip` archives for September 12–30, 2022, and each archive's `.CHECKSUM` file: recover observed daily candles missing from the stored price partitions. |
| `data/luna_event_review/` | `archive_manifest.json`, `ust_archive_manifest.json`, the four `LUNAUSDT-1m-2022-05-DD.zip` archives for May 10–13, 2022, `USTUSDT-1m-2022-05-13.zip`, and each archive's `.CHECKSUM` file: validate reviewed LUNA moves and supply observed LUNA/UST intraday exits. |
| `data/utk_event_review/` | `archive_manifest.json`, `UTKUSDT-1m-2026-04-14.zip`, and `UTKUSDT-1m-2026-04-14.zip.CHECKSUM`: supply the reviewed UTK intraday exit. |

Keep the ZIP archives compressed: the notebook reads them directly and checks them against their manifests and published checksums. Compressed price files and repair ZIP archives require their exact original byte checksums. Missing repair evidence is not fetched automatically; missing files or checksum mismatches require restoring the original local inputs. The daily-repair loader currently skips its repair step if its manifest is absent, so include that folder to reproduce the documented repaired dataset.

Python dependencies: NumPy and pandas; use Jupyter/IPython to run the notebook and display its tables and charts. Outputs are generated in `output/Common_Risk_Factors_Liu_et_al/`, so previous output folders are not required for a full run.

The Binance experiment audits 719 stored histories (718 exchange symbols plus the separate old-LUNA history). Of these, 668 could be matched successfully to market-cap data.


**Reporting conventions and alpha uncertainty.** Headline tables, rankings, charts, and unqualified `Sharpe` fields use **252-day Sharpe**, $\sqrt{252}\,\bar r/s_r$, with zero risk-free return. The main performance tables also report **Sharpe (365)** on the identical daily observations. This is a reporting convention: cryptocurrency observations remain calendar-daily, including weekends. CAGR, annual volatility, Sortino, turnover, linear annual alpha, borrowing accrual, and strategy risk sizing retain their documented 365-day calendar convention. Alpha is the intercept of the stated benchmark regression; its **HAC t-statistic and pointwise 95% confidence interval** accompany return results. Daily BTC regressions use 20 HAC lags, a finite-sample covariance correction, and Student-t critical values. Annual alpha and its bounds are daily estimates multiplied by 365 (and by 100 when displayed as percent), not compounded returns. Intervals crossing zero do not establish positive alpha. These intervals do not correct the prior strategy search or make development data unseen. **We do not change strategies to obtain a t-statistic above 2**; doing so would create further selection bias.

**Cost convention:** 20 bps per dollar traded is the baseline all-in transaction cost (approximately 7 bps commission plus 13 bps slippage). Legacy ledger fields named `fee` or `trading_fees` record this combined cost, not commission alone. Extra execution/borrowing cost scenarios are evaluated only as sensitivities in the final portfolio notebook.


## 1. Research design and experiment map

Compare CMC 30 and Binance matched, both cap-weighted, across the same nine long-short signals.
Set the fixed periods, 100,000 initial capital, 20-bps transaction fee and $1 million minimum cap.
`RUN_SCOPE='both'` evaluates backtest and validation; no equal-weighted or liquidity-screened
variant is constructed or simulated. Each experiment/strategy has its own independent account.
The next cell lists the fixed 30 CMC coins; Binance candidates come from the separate saved run.


In [1]:
from pathlib import Path
import hashlib, json, math, platform
import numpy as np
import pandas as pd
try:
    from IPython.display import display, HTML
except ImportError:
    def display(x): print(x.to_string() if hasattr(x, 'to_string') else x)
    def HTML(x): return x
# Notebook variables, change as needed
NOTEBOOK_NAME = 'Common_Risk_Factors_Liu_et_al.ipynb'
# Input and output paths are relative to the extracted repository folder.
# Open that folder as your IDE project and start the kernel there.
NOTEBOOK_DIR = Path(".")
DATA_DIR = Path('data')
COIN_ROOT = Path('Coin_Prices')
# Get correct data from binance downloads (Look at Data_Download_3731Coins.ipynb)
SOURCE_RUN = '20260918T174517572465Z'
RUN_DIR = COIN_ROOT / 'runs' / SOURCE_RUN
SAMPLE_START, SAMPLE_END = '2021-01-01', '2026-09-17'
BACKTEST_START, BACKTEST_END = '2021-01-02', '2024-12-31'
VALIDATION_START, VALIDATION_END = '2025-01-01', '2026-09-17'
# RUN_SCOPE can be backtest for just backtest, or both for backtest+validation
RUN_SCOPE = 'both'
assert RUN_SCOPE in {'backtest','both'}
DATA_END = BACKTEST_END if RUN_SCOPE == 'backtest' else SAMPLE_END
# Using data downloaded from cmc for the 30-coin universe
CMC_30_DIR = Path('OHLCV_Market_Cap_Data')
# BPS,initial capital,and cost rate
COST_BPS, INITIAL_CAPITAL = 20, 100_000.0
COST_RATE = COST_BPS / 10_000
MIN_MARKET_CAP, MIN_ELIGIBLE = 1_000_000.0, 10
PERIODS_PER_YEAR = 365
STRATEGIES = {'size':'Size', 'price':'Price', 'max_price':'Maximum close (7-day proxy)',
 'mom_1w':'Momentum 1 week', 'mom_2w':'Momentum 2 weeks', 'mom_3w':'Momentum 3 weeks',
 'mom_4w':'Momentum 4 weeks', 'dollar_volume':'Volume level', 'volume_volatility':'Volume volatility'}
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
# Checksums now tolerate only LF/CRLF differences against a frozen reference digest.
# Binary inputs always require their exact byte checksum; no whitespace/encoding rewriting.
def verify_checksum_bytes(data, expected_sha256, expected_lf_sha256=None, *, text=False):
    actual = hashlib.sha256(data).hexdigest()
    if actual == expected_sha256:
        return 'exact'
    if text and expected_lf_sha256 is not None:
        try:
            data.decode('utf-8')
        except UnicodeDecodeError:
            raise ValueError('Checksum mismatch: text is not valid UTF-8') from None
        canonical = hashlib.sha256(data.replace(b'\r\n', b'\n')).hexdigest()
        if canonical == expected_lf_sha256:
            return 'line_endings_only'
    raise ValueError('Checksum mismatch: file content differs from its reference')

source_hashes = {}
# Record the raw bytes received and whether text normalization was needed.
text_checksum_verification = {}
def record(path):
    source_hashes[Path(path).resolve().relative_to(NOTEBOOK_DIR.resolve()).as_posix()] = sha(path)
    return path
pd.set_option('display.max_rows', 50)
SHARPE_PERIODS_PER_YEAR = 252


def alpha_uncertainty(fit, periods=365):
    """Pointwise HAC alpha inference; annual bounds are linear percentages."""
    bounds = np.asarray(fit.conf_int(alpha=0.05))[0]
    return {"Alpha HAC t": float(np.asarray(fit.tvalues)[0]),
            "Alpha 95% CI low (%)": float(bounds[0] * periods * 100),
            "Alpha 95% CI high (%)": float(bounds[1] * periods * 100)}


def btc_alpha_metrics(r, benchmark):
    import statsmodels.api as sm
    aligned = pd.concat({"strategy": r, "BTC": benchmark.reindex(r.index)}, axis=1).dropna()
    if len(aligned) != len(r):
        raise ValueError("BTC alpha requires a benchmark observation for every account return date")
    fit = sm.OLS(aligned.strategy.to_numpy(), sm.add_constant(aligned.BTC.to_numpy())).fit(
        cov_type="HAC", cov_kwds={"maxlags": min(20, len(aligned)-1), "use_correction": True}, use_t=True)
    return {"Alpha vs BTC (%)": float(fit.params[0] * 36500),
            "Beta vs BTC": float(fit.params[1]), "Alpha HAC p": float(fit.pvalues[0]),
            **alpha_uncertainty(fit)}


In [2]:
# Fixed 30-coin universe, grouped by economic category
SECTORS = {
    "Store of Value / Benchmark": ["BTC-USD"],
    "Layer 1 Smart Contracts": ["ETH-USD", "SOL-USD", "BNB-USD", "ADA-USD", "AVAX-USD", "DOT-USD", "NEAR-USD", "ATOM-USD", "TRX-USD", "ALGO-USD", "HBAR-USD"],
    "Payments & Legacy": ["XRP-USD", "DOGE-USD", "LTC-USD", "BCH-USD", "ETC-USD", "XLM-USD"],
    "DeFi Protocols": ["AAVE-USD", "MKR-USD", "UNI7083-USD", "CRV-USD", "INJ-USD", "RUNE-USD"],
    "Infrastructure, Storage & Metaverse": ["LINK-USD", "FIL-USD", "VET-USD", "THETA-USD", "SAND-USD", "MANA-USD"]
}

# Flatten list of tickers
tickers = [t for sec in SECTORS.values() for t in sec]
START_DATE = "2021-01-01"
print (tickers)

print(f"Total Universe Size: {len(tickers)} cryptocurrencies")
for sec, coins in SECTORS.items():
    print(f"  • {sec}: {len(coins)} coins ({coins})")

# UNI7083 is the vendor alias for UNI; retain the same 30 economic assets.
ASSETS = sorted("UNI-USD" if t == "UNI7083-USD" else t for t in tickers)
assert len(ASSETS) == len(set(ASSETS)) == 30

['BTC-USD', 'ETH-USD', 'SOL-USD', 'BNB-USD', 'ADA-USD', 'AVAX-USD', 'DOT-USD', 'NEAR-USD', 'ATOM-USD', 'TRX-USD', 'ALGO-USD', 'HBAR-USD', 'XRP-USD', 'DOGE-USD', 'LTC-USD', 'BCH-USD', 'ETC-USD', 'XLM-USD', 'AAVE-USD', 'MKR-USD', 'UNI7083-USD', 'CRV-USD', 'INJ-USD', 'RUNE-USD', 'LINK-USD', 'FIL-USD', 'VET-USD', 'THETA-USD', 'SAND-USD', 'MANA-USD']
Total Universe Size: 30 cryptocurrencies
  • Store of Value / Benchmark: 1 coins (['BTC-USD'])
  • Layer 1 Smart Contracts: 11 coins (['ETH-USD', 'SOL-USD', 'BNB-USD', 'ADA-USD', 'AVAX-USD', 'DOT-USD', 'NEAR-USD', 'ATOM-USD', 'TRX-USD', 'ALGO-USD', 'HBAR-USD'])
  • Payments & Legacy: 6 coins (['XRP-USD', 'DOGE-USD', 'LTC-USD', 'BCH-USD', 'ETC-USD', 'XLM-USD'])
  • DeFi Protocols: 6 coins (['AAVE-USD', 'MKR-USD', 'UNI7083-USD', 'CRV-USD', 'INJ-USD', 'RUNE-USD'])
  • Infrastructure, Storage & Metaverse: 6 coins (['LINK-USD', 'FIL-USD', 'VET-USD', 'THETA-USD', 'SAND-USD', 'MANA-USD'])


## 2. CMC 30-coin inputs and provenance

Load the saved daily OHLCV and market-cap/supply files for the fixed 30-coin reference.
Validate identities, coverage, dates, OHLC bars and volume; preserve source labels, including the
five supplied historical Yahoo fallback rows. No Yahoo acquisition occurs. Daily caps determine
eligibility and within-leg weights for all nine signals. Missing/zero caps remain unavailable.


In [3]:
# Get data for 30-coin universe
cmc_source_audit = pd.DataFrame()
cmc_cap_issues = pd.DataFrame()
cmc_ids = {'AAVE':7278,'ADA':2010,'ALGO':4030,'ATOM':3794,'AVAX':5805,'BCH':1831,
    'BNB':1839,'BTC':1,'CRV':6538,'DOGE':74,'DOT':6636,'ETC':1321,'ETH':1027,
    'FIL':2280,'HBAR':4642,'INJ':7226,'LINK':1975,'LTC':2,'MANA':1966,'MKR':1518,
    'NEAR':6535,'RUNE':4157,'SAND':6210,'SOL':5426,'THETA':2416,'TRX':1958,
    'UNI':7083,'VET':3077,'XLM':512,'XRP':52}
cmc_ohlcv = pd.read_csv(record(CMC_30_DIR/'crypto_30_coin_daily_ohlcv.csv'))
cmc_cap_data = pd.read_csv(record(CMC_30_DIR/'crypto_30_coin_daily_market_cap_supply.csv'))
required_calendar = pd.date_range(SAMPLE_START, DATA_END, name='date')
expected_keys = pd.MultiIndex.from_product([required_calendar,sorted(cmc_ids)],names=['date','symbol'])
# Sanity checks
for label, panel in [('OHLCV',cmc_ohlcv),('caps',cmc_cap_data)]:
    panel['date'] = pd.to_datetime(panel.date,utc=True).dt.tz_localize(None)
    if not panel.date.eq(panel.date.dt.normalize()).all(): raise ValueError(f'{label}: dates must be UTC daily labels')
    if panel.duplicated(['date','symbol']).any(): raise ValueError(f'{label}: duplicate date/coin')
    if set(panel.symbol)!=set(cmc_ids): raise ValueError(f'{label}: unexpected 30-coin universe')
    if not panel.cmc_id.eq(panel.symbol.map(cmc_ids)).all(): raise ValueError(f'{label}: CMC identity mismatch')
    selected_keys = pd.MultiIndex.from_frame(panel.loc[panel.date.between(pd.Timestamp(SAMPLE_START),pd.Timestamp(DATA_END)),['date','symbol']])
    if len(expected_keys.difference(selected_keys)): raise ValueError(f'{label}: incomplete requested daily coverage')
cmc_ohlcv = cmc_ohlcv.loc[cmc_ohlcv.date.between(pd.Timestamp(SAMPLE_START),pd.Timestamp(DATA_END))].copy()
cmc_cap_data = cmc_cap_data.loc[cmc_cap_data.date.between(pd.Timestamp(SAMPLE_START),pd.Timestamp(DATA_END))].copy()
ohlc = cmc_ohlcv[['open','high','low','close']]
if not np.isfinite(ohlc).all().all() or ohlc.le(0).any().any(): raise ValueError('CMC prices must be finite and positive')
if (cmc_ohlcv.low.gt(ohlc[['open','close']].min(axis=1)) | cmc_ohlcv.high.lt(ohlc[['open','close']].max(axis=1))).any(): raise ValueError('Invalid CMC OHLC bar')
if not np.isfinite(cmc_ohlcv.volume).all() or cmc_ohlcv.volume.lt(0).any(): raise ValueError('Invalid CMC volume')
if not cmc_ohlcv.ohlcv_source.isin(['coinmarketcap','yahoo_finance_cmc_fallback']).all(): raise ValueError('Unknown CMC input provenance')
for field in ['market_cap','circulating_supply']:
    if np.isinf(cmc_cap_data[field]).any() or cmc_cap_data[field].lt(0).any(): raise ValueError(f'Invalid CMC {field}')
def cmc_panel(frame, field):
    result = frame.pivot(index='date',columns='symbol',values=field)
    result.columns = result.columns + '-USD'
    return result.reindex(index=required_calendar,columns=ASSETS).astype(float)
prices = cmc_panel(cmc_ohlcv,'close')
volume = cmc_panel(cmc_ohlcv,'volume')
cmc_30_caps = cmc_panel(cmc_cap_data,'market_cap')
# zero/missing market caps remain unavailable, never fabricated from prices or carried forward.
cmc_30_caps = cmc_30_caps.where(np.isfinite(cmc_30_caps) & cmc_30_caps.gt(0))
cmc_source_audit = cmc_ohlcv[['date','symbol','cmc_id','ohlcv_source','retrieved_at_utc']].copy()
cmc_cap_issues = cmc_cap_data.loc[cmc_cap_data.market_cap.isna() | cmc_cap_data.market_cap.le(0)].copy()
baseline_label = 'CMC 30 (labelled Yahoo fallback rows retained)'
baseline_experiment = 'cmc_30_cap'
returns = prices.pct_change(fill_method=None).iloc[1:]
display(cmc_source_audit.ohlcv_source.value_counts().rename('Rows by supplied source'))
display(cmc_cap_issues.groupby('symbol').size().rename('Unavailable daily caps'))

binance_config = json.loads(record(RUN_DIR/'config.json').read_text(encoding='utf8'))
binance_end = pd.Timestamp(binance_config['END_EXCLUSIVE']) - pd.Timedelta(days=1)
if binance_end < pd.Timestamp(DATA_END):
    raise ValueError(f'Binance coverage ends {binance_end.date()}; requested {DATA_END}')
print('Selected baseline:',baseline_label,'| requested end:',DATA_END)


ohlcv_source
coinmarketcap                 62575
yahoo_finance_cmc_fallback        5
Name: Rows by supplied source, dtype: int64

symbol
CRV       4
LINK      1
MKR     480
NEAR      1
Name: Unavailable daily caps, dtype: int64

Selected baseline: CMC 30 (labelled Yahoo fallback rows retained) | requested end: 2026-09-17


## 3. Binance prices and expanded historical market caps

### From 3,731 market records to 658 eligible histories

The starting catalogue contains exchange pairs, not 3,731 distinct coins. For the full configured sample, the extended universe is narrowed as follows:

| Step | Remaining histories | Explanation |
|---|---:|---|
| Exchange-pair catalogue | 3,731 | Start with the saved market records across all quote currencies. |
| USDT-quoted pairs | 746 | Keep USDT markets so the extended experiment uses one quote currency. |
| Successful daily downloads | 718 | Remove 24 histories without published rows and four failed downloads. |
| Separate old and new LUNA | 719 | Split the reused LUNA source symbol into two distinct token histories, adding one history. |
| Ordinary spot instruments | 676 | Exclude 40 leveraged products and three fiat pairs: AUD, EUR and GBP. |
| Usable historical market-cap matches | 668 | Remove eight histories without any usable historical cap match after identity and price-consistency checks. |
| Meet the $1 million cap floor at least once | **658** | Ten matched histories never reach the minimum cap, leaving 658 that qualify at least once. |

These **658 histories are candidates over the full sample**, not 658 simultaneous holdings: each weekly rebalance still requires a qualifying preceding-Sunday cap, sufficient valid signal history and trading availability. LUNA and the other reviewed event histories remain included where eligible, with the event handling described below.

Load the checksummed Binance partitions and preserve the existing candle repairs. The expanded local CMC weekly cache replaces the annual top-500 admission restriction. Current exchange-pair records establish identities only; their current caps are never used in historical portfolios. Each observation is dated to Sunday close and used at the following Monday rebalance.

The next section reports all 719 histories. Forty leveraged products remain excluded under the existing experiment design, as do fiat exchange-rate pairs. Unresolved identities, missing/nonpositive caps, incompatible prices, and duplicate coin identities on the same date remain unavailable rather than being filled. Scaled symbols such as 1000SATS use their unit multiplier only for the price cross-check, never to multiply market cap.

The 25% symmetric price-ratio limit is a predeclared integrity check, not tuned to strategy returns. It can conservatively reject genuine price dislocations. CMC histories are retrospective downloads and may contain provider revisions; this is not an archived point-in-time fundamentals database.

<!-- UPDATED: identify the locally loaded daily repair evidence. -->
Daily candle repairs are loaded from `data/backtest_repairs/archive_manifest.json` and its referenced daily ZIP/`.CHECKSUM` pairs. Historical caps are loaded from `data/binance_cap_expansion/weekly_observations.csv`; they are separate from the Binance price folder. All required local inputs are listed in the introduction. No repair download occurs in this section.

<!-- CHANGED: portable text checksums retain strict content verification. -->
The expanded-source manifest contains both original and LF-normalized SHA256 reference hashes for its CSV/JSON files. Verification accepts an exact hash match or a match after replacing CRLF with LF; it does not trim spaces, reorder rows, change numbers or remove an encoding marker. Actual raw hashes and any accepted line-ending differences are recorded in the output manifest and `text_checksum_verification.csv`. Compressed Binance price partitions and repair archives continue to require exact checksums.


In [4]:
# retain baseline inputs separately and load all 718 successful USDT markets.
calendar = pd.date_range(SAMPLE_START, DATA_END, freq='D', name='date')
baseline_prices, baseline_volume = prices.copy(), volume.copy()
# baseline caps come from the same CMC download when CMC is selected.
baseline_caps = cmc_30_caps.copy()
registry = pd.read_csv(record(RUN_DIR / 'partition_manifest.csv'))
instruments = pd.read_csv(record(RUN_DIR / 'instruments.csv'))
selected = registry.loc[registry.status.eq('ok') & registry.quote_asset.eq('USDT')].copy()
pair_audit = registry.copy()
pair_audit['decision'] = np.where(pair_audit.status.ne('ok'), 'unavailable partition',
                                  np.where(pair_audit.quote_asset.eq('USDT'), 'included USDT', 'other quote excluded'))
close_parts, volume_parts, partition_hashes = {}, {}, {}
for r in selected.itertuples():
    p = (RUN_DIR / r.path).resolve()
    if RUN_DIR.resolve() not in p.parents: raise ValueError('Unsafe partition path')
    digest = sha(p)
    if digest != r.sha256: raise ValueError(f'Partition checksum changed: {r.symbol}')
    partition_hashes[r.symbol] = digest
    frame = pd.read_csv(p, usecols=['date','close','quote_volume'])
    frame['date'] = pd.to_datetime(frame.date, utc=True).dt.tz_localize(None).dt.normalize()
    if frame.date.duplicated().any(): raise ValueError(f'Duplicate candles: {r.symbol}')
    frame = frame.set_index('date').reindex(calendar)
    close_parts[r.symbol] = frame.close.where(np.isfinite(frame.close) & frame.close.gt(0))
    volume_parts[r.symbol] = frame.quote_volume.where(np.isfinite(frame.quote_volume) & frame.quote_volume.ge(0))
binance_prices = pd.DataFrame(close_parts, index=calendar).sort_index(axis=1)
binance_volume = pd.DataFrame(volume_parts, index=calendar).reindex(columns=binance_prices.columns)


# repair incomplete monthly archives with independently checksummed daily archives.
# Only missing observations are filled; contradictory overlaps stop the run.
import zipfile
repair_audit = []
repair_root = DATA_DIR / 'backtest_repairs'
repair_manifest = repair_root / 'archive_manifest.json'
if repair_manifest.exists():
    for item in json.loads(record(repair_manifest).read_text(encoding='utf8')):
        if item['interval'] != '1d' or len(item['period']) != 10:
            continue
        asset = item['symbol']
        if asset not in binance_prices.columns: raise ValueError(f'Unknown repair asset: {asset}')
        archive = repair_root / item['archive']
        if archive.resolve().parent != repair_root.resolve(): raise ValueError('Unsafe repair path')
        if sha(record(archive)) != item['sha256']: raise ValueError('Repair archive hash mismatch')
        checksum = record(Path(str(archive) + '.CHECKSUM')).read_text().split()[0]
        if checksum != item['sha256']: raise ValueError('Published repair checksum mismatch')
        with zipfile.ZipFile(archive) as z:
            if len(z.namelist()) != 1: raise ValueError('Unexpected repair members')
            candles = pd.read_csv(z.open(z.namelist()[0]), header=None)
        if candles.shape != (1, 12): raise ValueError('Expected exactly one daily Binance candle')
        row = candles.iloc[0]
        day = pd.to_datetime(int(row[0]), unit='ms')
        if day != pd.Timestamp(item['period']): raise ValueError('Repair date mismatch')
        if int(row[6]) != int(row[0]) + 86_400_000 - 1: raise ValueError('Partial repair candle')
        o, h, l, c, v, q = [float(row[i]) for i in [1,2,3,4,5,7]]
        if not np.isfinite([o,h,l,c,v,q]).all() or min(o,h,l,c) <= 0 or min(v,q) < 0:
            raise ValueError('Invalid supplemental candle')
        if not l <= min(o,c) <= max(o,c) <= h: raise ValueError('Invalid repair OHLC')
        if day not in calendar: continue
        for field, panel, value in [('close',binance_prices,c),('quote_volume',binance_volume,q)]:
            old = panel.at[day, asset]
            if pd.notna(old) and not np.isclose(old,value,rtol=1e-10,atol=1e-10):
                raise ValueError(f'Conflicting repair: {asset} {day} {field}')
            if pd.isna(old):
                panel.at[day,asset] = value
                repair_audit.append({'asset':asset,'date':day,'field':field,'value':value,
                                     'source_url':item['url'],'sha256':item['sha256']})
repair_audit = pd.DataFrame(repair_audit)
print('Verified supplemental observations:', len(repair_audit))

# expand identity coverage beyond the unrelated annual top-500 admission list.
# Inputs are frozen locally; the notebook does not request live/current market caps.
expansion_root = DATA_DIR / 'binance_cap_expansion'
cap_expansion_manifest = json.loads(record(expansion_root/'manifest.json').read_text(encoding='utf8'))
for filename, expected_hash in cap_expansion_manifest['files'].items():
    source = expansion_root / filename
    if source.resolve().parent != expansion_root.resolve(): raise ValueError('Unsafe cap source path')
    expected_lf = cap_expansion_manifest.get('lf_normalized_sha256',{}).get(filename)
    try:
        verification = verify_checksum_bytes(source.read_bytes(),expected_hash,expected_lf,
            text=source.suffix.lower() in {'.csv','.json'})
    except ValueError as error:
        raise ValueError(f'Cap source checksum mismatch: {filename}; {error}') from error
    record(source)
    text_checksum_verification[filename] = {
        'verification':verification,'actual_sha256':sha(source),
        'expected_sha256':expected_hash,'expected_lf_sha256':expected_lf}
print('Text checksum verification:',pd.Series(
    [item['verification'] for item in text_checksum_verification.values()]).value_counts().to_dict())
cap_candidates = pd.read_csv(expansion_root/'identity_candidates.csv')
for column in ['valid_from','valid_to_exclusive']:
    cap_candidates[column] = pd.to_datetime(cap_candidates[column])
assert (cap_candidates.valid_from < cap_candidates.valid_to_exclusive).all()
assert cap_candidates.price_units.gt(0).all()
weekly = pd.read_csv(expansion_root/'weekly_observations.csv')
weekly['Date'] = pd.to_datetime(weekly.Date)
weekly['observed_at'] = pd.to_datetime(weekly.observed_at, utc=True)
assert not weekly.duplicated(['Date','CMC id']).any()
assert weekly.Date.dt.weekday.eq(6).all()
assert weekly.observed_at.dt.tz_localize(None).eq(weekly.Date + pd.Timedelta(days=1)-pd.Timedelta(milliseconds=1)).all()
weekly['valid_cap'] = weekly.market_cap.where(np.isfinite(weekly.market_cap) & weekly.market_cap.gt(0))
# The final identity/cap panels are constructed after splitting the reused LUNA ticker.
identity = pd.DataFrame(index=binance_prices.columns)
identity['base_asset'] = instruments.set_index('symbol').base_asset.reindex(identity.index)
binance_caps = pd.DataFrame(np.nan,index=calendar,columns=binance_prices.columns)
weekly_available_dates = pd.DatetimeIndex(weekly.Date.unique()).sort_values()
missing_weekly_dates = pd.date_range(SAMPLE_START,DATA_END,freq='W-SUN').difference(weekly_available_dates)
print('Expanded historical CMC inputs:',weekly['CMC id'].nunique(),'identities;',len(weekly),'weekly observations')


Verified supplemental observations: 38
Text checksum verification: {'exact': 51, 'line_endings_only': 4}
Expanded historical CMC inputs: 660 identities; 129123 weekly observations


## 4. Trading events, identity repairs and executable exits

Required local evidence: `data/verified_market_events.csv`, `data/binance_cap_expansion/additional_market_events.csv`, and the manifests, ZIP archives and `.CHECKSUM` files in `data/luna_event_review/` and `data/utk_event_review/`. These files must be supplied with the notebook; this section verifies and reads them locally rather than downloading them. See the introduction for the exact archive names.

The event table records Binance announcement timestamps, trading halts, reopening times, conversion
ratios and source links. At the first daily close after an announcement, close the affected position
and keep that cash until a normal rebalance. No new orders are allowed from that announcement until
the same symbol reopens; retired symbols remain blocked. The exit is allowed only before the announced
halt and at an observed price with positive turnover. Late announcements that leave no executable
daily close require intraday evidence; without it the status is execution_unavailable. No backwards search for a convenient exit price.

This deliberately uses an exit-before-event strategy instead of holding tokens through migrations.
It avoids inventing suspension-period marks or assuming identical short liabilities across token swaps.
Replacement symbols need their own observed history and CMC identity. Signals reset across the halt,
so reverse splits do not become momentum profits or losses. Conversion ratios are retained and tested
for value preservation, but do not inject proceeds into the account: known events have already been exited.

The registry covers verified events and is extendable; it is not a complete exchange-event database.
Unexplained held-price gaps still stop the account. Large observed daily price jumps are flagged for
review rather than silently clipped or treated as established insolvency. Source announcements may
have later edits; their archived publication vintages have not been independently reconstructed.
### Reviewed LUNA exception
<!-- CHANGED: narrow audited event handling, applied symmetrically to longs and shorts. -->
The May 11 and May 12, 2022 closing prices are corroborated by complete Binance one-minute
archives and published SHA256 checksums in data/luna_event_review. Those exact observations
may pass the extreme-move check. All other unreviewed jumps still stop for review.
No price, return, or account balance is clipped or reset.

For old LUNA, the model reacts one minute after the recorded May 13 00:28 UTC announcement,
then exits at the first full positive-turnover minute's close: 00:29:59.999 UTC at 0.00008 USDT,
before the 00:40 UTC halt. Costs apply to actual exit notional. Exit size must be at most
1% of that minute's quote turnover. This is a stated fill assumption using observed trades,
not a guarantee of execution or verification of short borrow availability. Borrowing costs,
margin constraints, and archived announcement vintages remain limitations.

Old LUNA is represented by LUNA_CLASSIC_USDT. Terra 2.0 keeps LUNAUSDT but starts with the
first full UTC day on June 1, 2022. LUNCUSDT remains a separate observed market: no holdings,
indicators, or new-token airdrops are carried across these identities.
[Halt notice](https://www.binance.com/en/support/announcement/detail/514e0ba636e843bda47d5d740b7dadf4) |
[Terra 2.0 and historical ticker notice](https://www.binance.com/en-IN/support/announcement/detail/d044a6742e484b77a170111460b0eed3).

The same notice halted UST at 00:50 UTC. Its independently checksum-verified May 13 minute
archive supplies a UST exit using the identical one-minute reaction rule, participation limit
and transaction cost. This prevents a second unsupported daily-close exit in the same portfolios.

### Additional reviewed trading events
<!-- CHANGED: repair missing event coverage without removing historical constituents. -->
The event table includes the [AUD pair removal](https://www.binance.com/en/support/announcement/detail/bd9a8eb9e0c940278d888cb95b76ba41),
[TVK to VANRY](https://www.binance.com/en/support/announcement/detail/1493a2222fa549ce93f7101abe0941bf),
[TOMO to VIC](https://www.binance.com/az-AZ/support/announcement/detail/034eee2293964e26b915b1ec5c42972a), and
[STRAX redenomination](https://www.binance.com/en/support/announcement/detail/2e6eb8e644664a6ca7b6659a226c41b0).
Positions exit at the first daily close after each published announcement, before trading stops.
Observed prices and transaction costs apply. Historical constituents remain in the universe;
replacement tickers are not spliced, and STRAX signals reset across its suspension.
These are retrospective event-data repairs, not evidence of profitable or prospective performance.
The same verified event and identity rules are retained for the Binance experiment. Records for
ineligible instruments remain in the evidence table, without introducing additional experiments.

### Reviewed UTK intraday exit
<!-- CHANGED: resolve the unsupported same-day halt with observed minute evidence. -->
Binance announced UTK removal on April 14, 2026 at 02:00 UTC, with spot trading stopping at
14:00 UTC that day. There is no executable daily close after that announcement. The local,
checksum-verified minute archive supplies an exit using the existing one-minute reaction rule:
the first positive-turnover minute opens at 02:01 and closes at 02:01:59.999999, at
0.01433 USDT, with 223,043.10102 USDT of turnover. The existing 1% participation
limit allows at most 2,230.4310102 USDT per simulated account in that minute; larger positions
still stop. Costs apply to actual exit notional, and no XMN swap or airdrop value is credited.
The archive uses microsecond timestamps and contains all 840 minutes before the known halt.
This is an observed-price fill assumption, not proof of short borrowing or guaranteed execution.
[Binance notice](https://www.binance.com/en/support/announcement/detail/895c114d247a41159cc7e4c913f42c36).


Identity repair: old/new LUNA and reviewed redenominations use separate date-bounded CMC identities. Renamed exchange symbols retain separate price histories. Two symbols cannot receive the same coin cap on the same Sunday. The exported identity audit distinguishes an identified instrument from one with usable cap observations; the detailed cap audit records rejections.

The expanded universe also requires the announced BEAM delisting and KEY suspension/relisting. KEY was not tradable in USDT between February 10 and March 10, 2023: this is an event gap, not missing candles to interpolate. The same announcement-driven exit and signal-reset rules apply.


In [5]:
# Source-backed event schedule. UTC timestamps are stored timezone-naive consistently here.
# The event CSV includes AUD removal, TVK/VANRY and TOMO/VIC migrations, and STRAX redenomination.
# Apply the same announcement-day exit rule; retain earlier history and do not stitch tokens.
# Includes verified BEAM retirement and KEY suspension/relisting.
events = pd.concat([pd.read_csv(record(DATA_DIR / 'verified_market_events.csv')),
    pd.read_csv(record(expansion_root/'additional_market_events.csv'))],ignore_index=True)
for col in ['announced_at','halt_at','resume_at']:
    events[col] = pd.to_datetime(events[col], errors='raise')
assert events.announced_at.notna().all() and events.halt_at.notna().all()
assert (events.announced_at < events.halt_at).all()
assert not events.duplicated(['symbol','halt_at']).any()
assert events.source_url.str.startswith('https://www.binance.com/').all()


# Distinct instruments for old LUNA (later LUNC) and Terra 2.0.
# These internal labels never imply a quantity conversion or an airdrop entitlement.
LUNA_OLD = 'LUNA_CLASSIC_USDT'
LUNA_NEW_START = pd.Timestamp('2022-06-01')  # first full UTC day after May 31 06:00 listing
LUNA_HALT = pd.Timestamp('2022-05-13 00:40')
LUNA_ANNOUNCED = pd.Timestamp('2022-05-13 00:28')
LUNA_IDENTITY_SOURCE = 'https://www.binance.com/en-IN/support/announcement/detail/d044a6742e484b77a170111460b0eed3'
for frame in [binance_prices, binance_volume]:
    original_luna = frame['LUNAUSDT'].copy()
    frame[LUNA_OLD] = original_luna.where(frame.index < pd.Timestamp('2022-05-14'))
    frame['LUNAUSDT'] = original_luna.where(frame.index >= LUNA_NEW_START)
# Initialize the split histories; date-specific cap identities are assigned below.
binance_caps = binance_caps.reindex(columns=binance_prices.columns)
binance_caps['LUNAUSDT'] = np.nan
identity.loc[LUNA_OLD] = identity.loc['LUNAUSDT'].copy()
identity.loc[LUNA_OLD, 'base_asset'] = 'LUNA_CLASSIC'
luna_event = events.symbol.eq('LUNAUSDT') & events.halt_at.eq(LUNA_HALT)
assert luna_event.sum() == 1
events.loc[luna_event, 'symbol'] = LUNA_OLD
events.loc[luna_event, 'replacement'] = ''
events.loc[luna_event, 'resume_at'] = pd.NaT  # the OLD instrument never resumes as Terra 2.0
events.loc[luna_event, 'event_type'] = 'old_luna_retirement'
luna_identity_audit = pd.DataFrame([
    {'instrument':LUNA_OLD,'exchange_symbol':'LUNAUSDT','identity':'Terra Classic, old LUNA',
     'last_trading_timestamp':str(LUNA_HALT),'source_url':LUNA_IDENTITY_SOURCE},
    {'instrument':'LUNAUSDT','exchange_symbol':'LUNAUSDT','identity':'Terra 2.0',
     'first_full_daily_bar':str(LUNA_NEW_START.date()),'source_url':LUNA_IDENTITY_SOURCE},
])
binance_prices = binance_prices.sort_index(axis=1)
binance_volume = binance_volume.reindex(columns=binance_prices.columns)
binance_caps = binance_caps.reindex(columns=binance_prices.columns)

# Exact product identities avoid excluding ordinary tokens such as JUP and SYRUP.
# Binance explicitly identifies these 40 UP/DOWN products in this table:
PRODUCT_SOURCE = 'https://www.binance.com/en/support/announcement/detail/023a29887bdb47c9b55901cd1bb8eec9'
leveraged_underlyings = 'LTC BCH UNI XLM SXP ADA XTZ DOT ETH YFI EOS BTC 1INCH XRP SUSHI FIL LINK TRX BNB AAVE'.split()
leveraged_symbols = {u+side+'USDT' for u in leveraged_underlyings for side in ['UP','DOWN']}
product_audit = pd.DataFrame({'symbol':binance_prices.columns})
product_audit['product_class'] = np.where(product_audit.symbol.isin(leveraged_symbols),'leveraged_token','ordinary_spot')
product_audit['source_url'] = np.where(product_audit.symbol.isin(leveraged_symbols), PRODUCT_SOURCE, '')
fiat_symbols = {'AUDUSDT','EURUSDT','GBPUSDT'}
product_audit.loc[product_audit.symbol.isin(fiat_symbols),'product_class'] = 'fiat_currency'
ordinary = product_audit.loc[product_audit.product_class.eq('ordinary_spot'),'symbol'].tolist()
assert 'JUPUSDT' in ordinary and 'SYRUPUSDT' in ordinary

# Audit all 719 histories, then match only exact-date, usable cap observations.
# A 25% symmetric price-ratio tolerance is a data-integrity check, not a return filter.
# Identity evidence must exist first; similar prices alone never establish an identity.
CAP_PRICE_RATIO_LIMIT = 1.25
cap_checks = []
weekly_groups = {int(cid): group.set_index('Date') for cid,group in weekly.groupby('CMC id')}
for candidate in cap_candidates.itertuples():
    if candidate.asset not in ordinary: continue
    asset_prices = binance_prices[candidate.asset]
    dates = calendar[(calendar.weekday == 6) & (calendar >= candidate.valid_from) &
                     (calendar < candidate.valid_to_exclusive) & asset_prices.notna().to_numpy()]
    observed = weekly_groups.get(candidate.cmc_id,pd.DataFrame(columns=weekly.columns)).reindex(dates)
    check = pd.DataFrame({'date':dates,'asset':candidate.asset,'cmc_id':candidate.cmc_id,
        'binance_close':asset_prices.reindex(dates).to_numpy(),
        'cmc_close':pd.to_numeric(observed['close'],errors='coerce').to_numpy(),
        'market_cap':pd.to_numeric(observed['valid_cap'],errors='coerce').to_numpy(),
        'price_units':candidate.price_units,'basis':candidate.basis,'source':candidate.source})
    check['price_ratio'] = check.cmc_close * candidate.price_units / check.binance_close
    check['decision'] = np.select([
        ~np.isfinite(check.market_cap) | check.market_cap.le(0),
        ~np.isfinite(check.price_ratio) | ~check.price_ratio.between(1/CAP_PRICE_RATIO_LIMIT,CAP_PRICE_RATIO_LIMIT)
        ],['missing_or_nonpositive_cap','price_identity_check_failed'],default='accepted')
    cap_checks.append(check)
cap_match_audit = pd.concat(cap_checks,ignore_index=True)
# No arbitrary choice when two candidates or aliases claim the same economic coin/week.
accepted = cap_match_audit.decision.eq('accepted')
duplicate_asset = cap_match_audit.loc[accepted].duplicated(['date','asset'],keep=False)
duplicate_coin = cap_match_audit.loc[accepted].duplicated(['date','cmc_id'],keep=False)
conflicting = duplicate_asset | duplicate_coin
cap_match_audit.loc[conflicting.index[conflicting],'decision'] = 'ambiguous_or_duplicate_coin_week'
valid_matches = cap_match_audit.loc[cap_match_audit.decision.eq('accepted')]
assert not valid_matches.duplicated(['date','asset']).any()
assert not valid_matches.duplicated(['date','cmc_id']).any()
binance_caps = valid_matches.pivot(index='date',columns='asset',values='market_cap').reindex(
    index=calendar,columns=binance_prices.columns)
assert binance_caps.loc[calendar.weekday != 6].isna().all().all()  # never carry caps forward
assert binance_caps[sorted(leveraged_symbols | fiat_symbols)].isna().all().all()
# Counts describe usable observations, not a constant membership over the entire test.
identity['candidate_cmc_ids'] = cap_candidates.groupby('asset').cmc_id.agg(lambda x:'|'.join(map(str,sorted(set(x)))))
identity['cap_weeks'] = binance_caps.notna().sum()
identity['weeks_above_minimum_cap'] = binance_caps.ge(MIN_MARKET_CAP).sum()
identity['mapping_status'] = np.select([
    identity.index.isin(leveraged_symbols),identity.index.isin(fiat_symbols),
    identity.candidate_cmc_ids.isna(),identity.cap_weeks.eq(0)],
    ['excluded_leveraged_product','excluded_fiat_currency','unresolved_identity','mapped_no_usable_cap_history'],
    default='matched_with_usable_cap_history')
identity['first_usable_cap'] = binance_caps.apply(lambda x:x.first_valid_index())
identity['last_usable_cap'] = binance_caps.apply(lambda x:x.last_valid_index())
identity.index.name = 'asset'
matched = identity.cap_weeks.gt(0)
input_summary = pd.DataFrame([
    {'Input':baseline_label,'Assets':baseline_prices.shape[1],'Rows':baseline_prices.notna().sum().sum()},
    {'Input':'Binance histories audited (including split LUNA)','Assets':len(identity),'Rows':binance_prices.notna().sum().sum()},
    {'Input':'Binance histories with candidate identities','Assets':identity.candidate_cmc_ids.notna().sum(),'Rows':len(cap_candidates)},
    {'Input':'Binance histories with usable historical caps','Assets':matched.sum(),'Rows':len(valid_matches)}])
display(input_summary)
display(identity.mapping_status.value_counts().rename('Histories'))
display(identity.loc[~matched,['candidate_cmc_ids','mapping_status']])

def compile_event_policy(index, columns, event_table):
    blocked = pd.DataFrame(False,index=index,columns=columns)
    can_execute = pd.DataFrame(True,index=index,columns=columns)
    reset_signal = pd.DataFrame(False,index=index,columns=columns)
    closes = index + pd.Timedelta(days=1)  # close is the end of the labelled UTC day
    for event in event_table.itertuples():
        if event.symbol not in columns: continue
        same_symbol = event.replacement == event.symbol
        resume = event.resume_at.ceil('D') if same_symbol and pd.notna(event.resume_at) else pd.Timestamp('2262-04-11')
        blocked.loc[(closes > event.announced_at) & (index < resume),event.symbol] = True
        # A partial daily candle on halt day is not an executable midnight close.
        suspended = (closes >= event.halt_at) & (index < resume)
        can_execute.loc[suspended,event.symbol] = False
        reset_signal.loc[suspended,event.symbol] = True
        if pd.notna(event.resume_at) and pd.notna(event.replacement) and event.replacement != event.symbol and event.replacement in columns:
            # Do not trade a new ticker's partial reopening day or splice it to an old ticker.
            pending = (closes > event.announced_at) & (index < event.resume_at.ceil('D'))
            can_execute.loc[pending,event.replacement] = False
            reset_signal.loc[pending,event.replacement] = True
    return {'blocked':blocked, 'can_execute':can_execute, 'reset_signal':reset_signal}

event_policy = compile_event_policy(calendar,binance_prices.columns,events)

# Binance's published SHA256 checksums authenticate these cached minute archives.
import zipfile
LUNA_EVIDENCE_DIR = DATA_DIR / 'luna_event_review'
luna_archive_manifest = json.loads(record(LUNA_EVIDENCE_DIR / 'archive_manifest.json').read_text(encoding='utf8'))
def validate_complete_minute_day(opened, day):
    observed = pd.DatetimeIndex(opened).as_unit('ns')
    expected = pd.date_range(day, periods=1440, freq='min').as_unit('ns')
    if not observed.equals(expected):
        raise ValueError(
            f"Incomplete or misaligned minute archive for {day}: "
            f"expected 1440 ordered UTC minute opens, got {len(observed)}; "
            f"missing={len(expected.difference(observed))}, "
            f"unexpected={len(observed.difference(expected))}, "
            f"duplicates={int(observed.duplicated().sum())}. "
            "Review the source archive; do not fill or skip missing bars.")

luna_minutes = {}
for entry in luna_archive_manifest:
    archive = record(LUNA_EVIDENCE_DIR / entry['archive'])
    checksum = record(LUNA_EVIDENCE_DIR / (entry['archive'] + '.CHECKSUM'))
    assert sha(archive) == entry['sha256'] == checksum.read_text().split()[0], 'LUNA evidence changed'
    with zipfile.ZipFile(archive) as z:
        assert len(z.namelist()) == 1
        minute = pd.read_csv(z.open(z.namelist()[0]), header=None)
    minute.columns = ['open_time','open','high','low','close','volume','close_time',
                      'quote_volume','trades','taker_base','taker_quote','ignore']
    minute['opened'] = pd.to_datetime(minute.open_time, unit='ms')
    minute['closed'] = pd.to_datetime(minute.close_time, unit='ms')
    assert minute.opened.is_unique and minute.opened.is_monotonic_increasing
    assert (minute.closed - minute.opened).eq(pd.Timedelta(milliseconds=59999)).all()
    assert (minute['low'] <= minute[['open','close']].min(axis=1)).all()
    assert (minute['high'] >= minute[['open','close']].max(axis=1)).all()
    assert np.isfinite(minute[['open','high','low','close','quote_volume']]).all().all()
    assert minute['low'].gt(0).all() and minute.quote_volume.ge(0).all()
    if entry['date'] != '2022-05-13':
        validate_complete_minute_day(minute.opened, entry['date'])
    luna_minutes[entry['date']] = minute

luna_reviews = pd.DataFrame([
    {'date':pd.Timestamp('2022-05-11'),'instrument':LUNA_OLD,'previous_close':17.46,'close':1.0769},
    {'date':pd.Timestamp('2022-05-12'),'instrument':LUNA_OLD,'previous_close':1.0769,'close':0.00032},
])
for review in luna_reviews.itertuples():
    for day, expected in [(review.date-pd.Timedelta(days=1),review.previous_close),(review.date,review.close)]:
        actual = float(luna_minutes[str(day.date())].iloc[-1]['close'])
        assert np.isclose(actual, expected, rtol=1e-10, atol=1e-12), 'Reviewed minute price changed'
        if day in binance_prices.index:
            assert np.isclose(binance_prices.at[day,LUNA_OLD], expected, rtol=1e-10, atol=1e-12), 'Daily price no longer matches reviewed evidence'
luna_reviews['return'] = luna_reviews['close']/luna_reviews.previous_close-1
luna_reviews['review_basis'] = 'Genuine price collapse; daily close corroborated by complete checksum-verified minute bars'
luna_reviews['source_url'] = 'https://data.binance.vision/?prefix=data/spot/daily/klines/LUNAUSDT/1m/'
event_policy['verified_extreme'] = pd.DataFrame(False,index=calendar,columns=binance_prices.columns)
for review in luna_reviews.itertuples():
    if review.date in calendar: event_policy['verified_extreme'].at[review.date,LUNA_OLD] = True

# one-minute reaction delay, then the FIRST full positive-turnover bar before the halt.
# No search for the best exit price; the same rule applies to long positions and short liabilities.
exit_candidates = luna_minutes['2022-05-13']
exit_candidates = exit_candidates.loc[
    exit_candidates.opened.ge(LUNA_ANNOUNCED + pd.Timedelta(minutes=1)) &
    exit_candidates.closed.lt(LUNA_HALT) & exit_candidates.quote_volume.gt(0)]
assert not exit_candidates.empty, 'No observed executable LUNA minute before halt'
luna_exit = exit_candidates.iloc[0]
assert luna_exit.opened == pd.Timestamp('2022-05-13 00:29')
luna_exit_evidence = pd.DataFrame([{
    'instrument':LUNA_OLD,'announced_at':LUNA_ANNOUNCED,'halt_at':LUNA_HALT,
    'opened':luna_exit.opened,'executed_at':luna_exit.closed,'price':float(luna_exit['close']),
    'quote_volume':float(luna_exit.quote_volume),
    'source_url':'https://data.binance.vision/data/spot/daily/klines/LUNAUSDT/1m/LUNAUSDT-1m-2022-05-13.zip',
    'notice_url':'https://www.binance.com/en/support/announcement/detail/514e0ba636e843bda47d5d740b7dadf4',
}])
for name in ['intraday_exit_price','intraday_exit_volume']:
    event_policy[name] = pd.DataFrame(np.nan,index=calendar,columns=binance_prices.columns)
event_policy['intraday_exit_time'] = pd.DataFrame(pd.NaT,index=calendar,columns=binance_prices.columns)
exit_day = luna_exit.opened.normalize()
if exit_day in calendar:
    event_policy['intraday_exit_price'].at[exit_day,LUNA_OLD] = float(luna_exit['close'])
    event_policy['intraday_exit_volume'].at[exit_day,LUNA_OLD] = float(luna_exit.quote_volume)
    event_policy['intraday_exit_time'].at[exit_day,LUNA_OLD] = luna_exit.closed

# UST was also held by these portfolios and halted under the same notice.
ust_manifest = json.loads(record(LUNA_EVIDENCE_DIR/'ust_archive_manifest.json').read_text(encoding='utf8'))
ust_archive = record(LUNA_EVIDENCE_DIR/ust_manifest['archive'])
ust_checksum = record(LUNA_EVIDENCE_DIR/(ust_manifest['archive']+'.CHECKSUM'))
assert sha(ust_archive)==ust_manifest['sha256']==ust_checksum.read_text().split()[0]
with zipfile.ZipFile(ust_archive) as z:
    assert len(z.namelist())==1
    ust_minute = pd.read_csv(z.open(z.namelist()[0]),header=None)
ust_minute.columns = ['open_time','open','high','low','close','volume','close_time','quote_volume','trades','taker_base','taker_quote','ignore']
ust_minute['opened']=pd.to_datetime(ust_minute.open_time,unit='ms')
ust_minute['closed']=pd.to_datetime(ust_minute.close_time,unit='ms')
assert ust_minute.opened.is_unique and ust_minute.opened.is_monotonic_increasing
assert (ust_minute.closed-ust_minute.opened).eq(pd.Timedelta(milliseconds=59999)).all()
assert np.isfinite(ust_minute[['open','high','low','close','quote_volume']]).all().all()
assert ust_minute['low'].gt(0).all() and ust_minute.quote_volume.ge(0).all()
assert (ust_minute['low']<=ust_minute[['open','close']].min(axis=1)).all()
assert (ust_minute['high']>=ust_minute[['open','close']].max(axis=1)).all()
ust_event=events.loc[events.symbol.eq('USTUSDT') & events.halt_at.eq(pd.Timestamp('2022-05-13 00:50'))]
assert len(ust_event)==1
ust_event=ust_event.iloc[0]
ust_candidates=ust_minute.loc[
    ust_minute.opened.ge(ust_event.announced_at+pd.Timedelta(minutes=1)) &
    ust_minute.closed.lt(ust_event.halt_at) & ust_minute.quote_volume.gt(0)]
assert not ust_candidates.empty
ust_exit=ust_candidates.iloc[0]
assert ust_exit.opened==pd.Timestamp('2022-05-13 00:29')
if exit_day in calendar and 'USTUSDT' in binance_prices.columns:
    event_policy['intraday_exit_price'].at[exit_day,'USTUSDT']=float(ust_exit['close'])
    event_policy['intraday_exit_volume'].at[exit_day,'USTUSDT']=float(ust_exit.quote_volume)
    event_policy['intraday_exit_time'].at[exit_day,'USTUSDT']=ust_exit.closed
luna_exit_evidence=pd.concat([luna_exit_evidence,pd.DataFrame([{
    'instrument':'USTUSDT','announced_at':ust_event.announced_at,'halt_at':ust_event.halt_at,
    'opened':ust_exit.opened,'executed_at':ust_exit.closed,'price':float(ust_exit['close']),
    'quote_volume':float(ust_exit.quote_volume),'source_url':ust_manifest['source_url'],
    'notice_url':ust_event.source_url,
}])],ignore_index=True)

display(luna_reviews)
display(luna_exit_evidence)

# CHANGED: verified UTK same-day exit, using the existing one-minute reaction rule.
# Read local Binance evidence only; no inferred delisting proceeds or token-swap credit.
UTK_EVIDENCE_DIR = DATA_DIR / 'utk_event_review'
utk_manifest = json.loads(record(UTK_EVIDENCE_DIR/'archive_manifest.json').read_text(encoding='utf8'))
utk_archive = record(UTK_EVIDENCE_DIR/utk_manifest['archive'])
utk_checksum = record(UTK_EVIDENCE_DIR/(utk_manifest['archive']+'.CHECKSUM'))
assert sha(utk_archive) == utk_manifest['sha256'] == utk_checksum.read_text().split()[0]
with zipfile.ZipFile(utk_archive) as z:
    assert z.namelist() == ['UTKUSDT-1m-2026-04-14.csv']
    utk_minutes = pd.read_csv(z.open(z.namelist()[0]),header=None)
utk_minutes.columns = ['open_time','open','high','low','close','volume','close_time',
                       'quote_volume','trades','taker_base','taker_quote','ignore']
# Binance 2026 spot archives use MICROseconds, unlike the 2022 LUNA millisecond files.
assert utk_manifest['timestamp_unit'] == 'us'
utk_minutes['opened'] = pd.to_datetime(utk_minutes.open_time,unit='us')
utk_minutes['closed'] = pd.to_datetime(utk_minutes.close_time,unit='us')
utk_event = events.loc[events.symbol.eq('UTKUSDT') & events.halt_at.eq(pd.Timestamp('2026-04-14 14:00'))]
assert len(utk_event) == 1
utk_event = utk_event.iloc[0]
assert utk_event.announced_at == pd.Timestamp('2026-04-14 02:00')
# This is a partial trading day by design: require every minute up to the known halt.
utk_expected = pd.date_range('2026-04-14',utk_event.halt_at,freq='min',inclusive='left')
assert pd.DatetimeIndex(utk_minutes.opened).as_unit('ns').equals(utk_expected.as_unit('ns'))
# CHANGED: the venue's final pre-halt candle ends at millisecond precision.
assert (utk_minutes.closed.iloc[:-1]-utk_minutes.opened.iloc[:-1]).eq(pd.Timedelta(minutes=1)-pd.Timedelta(microseconds=1)).all()
assert utk_minutes.closed.iloc[-1] == utk_event.halt_at-pd.Timedelta(milliseconds=1)
assert np.isfinite(utk_minutes[['open','high','low','close','volume','quote_volume']]).all().all()
assert utk_minutes[['open','high','low','close']].gt(0).all().all()
assert utk_minutes[['volume','quote_volume']].ge(0).all().all()
assert (utk_minutes.low<=utk_minutes[['open','close']].min(axis=1)).all()
assert (utk_minutes.high>=utk_minutes[['open','close']].max(axis=1)).all()
utk_candidates = utk_minutes.loc[
    utk_minutes.opened.ge(utk_event.announced_at+pd.Timedelta(minutes=1))
    & utk_minutes.closed.lt(utk_event.halt_at) & utk_minutes.quote_volume.gt(0)]
assert not utk_candidates.empty
utk_exit = utk_candidates.iloc[0]  # first eligible bar; never choose the best later price
assert utk_exit.opened == pd.Timestamp('2026-04-14 02:01')
utk_exit_day = utk_exit.opened.normalize()
if utk_exit_day in calendar:
    event_policy['intraday_exit_price'].at[utk_exit_day,'UTKUSDT'] = float(utk_exit['close'])
    event_policy['intraday_exit_volume'].at[utk_exit_day,'UTKUSDT'] = float(utk_exit.quote_volume)
    event_policy['intraday_exit_time'].at[utk_exit_day,'UTKUSDT'] = utk_exit.closed
utk_exit_evidence = pd.DataFrame([{
    'instrument':'UTKUSDT','announced_at':utk_event.announced_at,'halt_at':utk_event.halt_at,
    'opened':utk_exit.opened,'executed_at':utk_exit.closed,'price':float(utk_exit['close']),
    'quote_volume':float(utk_exit.quote_volume),'maximum_fill_notional':.01*float(utk_exit.quote_volume),
    'source_url':utk_manifest['source_url'],'notice_url':utk_event.source_url,
    'archive_sha256':utk_manifest['sha256'],
}])
display(utk_exit_evidence)

signal_prices = binance_prices.mask(event_policy['reset_signal'])
signal_volume = binance_volume.mask(event_policy['reset_signal'])
def select_policy(columns):
    return {k:v.reindex(columns=columns) for k,v in event_policy.items()}
def converted_value(units, old_price, new_units_per_old):
    # Mechanical conversion only, independent of any actual economic price movement.
    return (units*new_units_per_old)*(old_price/new_units_per_old)
display(events[['symbol','announced_at','halt_at','replacement','new_units_per_old']])
display(product_audit.product_class.value_counts().rename('Markets'))

,Input,Assets,Rows
0,CMC 30 (labelled Yahoo fallback rows retained),30,62580
1,Binance histories audited (including split LUNA),719,754201
2,Binance histories with candidate identities,675,683
3,Binance histories with usable historical caps,668,102813


mapping_status
matched_with_usable_cap_history    668
excluded_leveraged_product          40
mapped_no_usable_cap_history         7
excluded_fiat_currency               3
unresolved_identity                  1
Name: Histories, dtype: int64

,candidate_cmc_ids,mapping_status
asset,,
1INCHDOWNUSDT,NaN,excluded_leveraged_product
1INCHUPUSDT,NaN,excluded_leveraged_product
AAVEDOWNUSDT,NaN,excluded_leveraged_product
AAVEUPUSDT,NaN,excluded_leveraged_product
ADADOWNUSDT,NaN,excluded_leveraged_product
...,...,...
XRPUPUSDT,NaN,excluded_leveraged_product
XTZDOWNUSDT,NaN,excluded_leveraged_product
XTZUPUSDT,NaN,excluded_leveraged_product


,date,instrument,previous_close,close,return,review_basis,source_url
0,2022-05-11,LUNA_CLASSIC_USDT,17.4600,1.07690,-0.938322,Genuine price collapse; daily close corroborat...,https://data.binance.vision/?prefix=data/spot/...
1,2022-05-12,LUNA_CLASSIC_USDT,1.0769,0.00032,-0.999703,Genuine price collapse; daily close corroborat...,https://data.binance.vision/?prefix=data/spot/...


,instrument,announced_at,halt_at,opened,executed_at,price,quote_volume,source_url,notice_url
0,LUNA_CLASSIC_USDT,2022-05-13 00:28:00,2022-05-13 00:40:00,2022-05-13 00:29:00,2022-05-13 00:29:59.999,0.000080,351586.879011,https://data.binance.vision/data/spot/daily/kl...,https://www.binance.com/en/support/announcemen...
1,USTUSDT,2022-05-13 00:28:00,2022-05-13 00:50:00,2022-05-13 00:29:00,2022-05-13 00:29:59.999,0.302903,747944.811912,https://data.binance.vision/data/spot/daily/kl...,https://www.binance.com/en/support/announcemen...


,instrument,announced_at,halt_at,opened,executed_at,price,quote_volume,maximum_fill_notional,source_url,notice_url,archive_sha256
0,UTKUSDT,2026-04-14 02:00:00,2026-04-14 14:00:00,2026-04-14 02:01:00,2026-04-14 02:01:59.999999,0.01433,223043.10102,2230.43101,https://data.binance.vision/data/spot/daily/kl...,https://www.binance.com/en/support/announcemen...,d4621947cd72992322a274711776414e6708d007bfd0fa...


,symbol,announced_at,halt_at,replacement,new_units_per_old
0,PAXUSDT,2021-08-30 08:11:00,2021-09-06 04:00:00,USDPUSDT,1.00
1,LITUSDT,2025-01-22 09:00:00,2025-02-10 03:00:00,HEIUSDT,1.00
2,BNXUSDT,2025-02-26 02:00:00,2025-03-18 03:00:00,FORMUSDT,1.00
3,BTCSTUSDT,2021-03-11 09:52:00,2021-03-15 07:00:00,BTCSTUSDT,10.00
4,DREPUSDT,2021-03-22 05:00:00,2021-03-29 04:00:00,DREPUSDT,0.01
...,...,...,...,...,...
199,TVKUSDT,2023-11-17 03:30:00,2023-11-27 03:00:00,VANRYUSDT,1.00
200,TOMOUSDT,2023-11-06 09:30:00,2023-11-20 03:00:00,VICUSDT,1.00
201,STRAXUSDT,2024-03-08 08:30:00,2024-03-20 03:00:00,STRAXUSDT,10.00
202,BEAMUSDT,2023-01-19 04:00:00,2023-01-26 09:00:00,NaN,NaN


product_class
ordinary_spot      676
leveraged_token     40
fiat_currency        3
Name: Markets, dtype: int64

## 5. Signals, weekly portfolios and capital accounting
<!-- CHANGED: shared engine for changing universes and separate accounts. -->
Signals retain the original definitions: small size, low nominal price, low maximum seven-day
close, high 1–4 week momentum, low seven-day mean volume, and low seven-day volume standard deviation.
All rolling windows require complete observations. Volume factors require positive daily volume.
Binance volume means venue quote turnover; the CMC baseline uses the supplied dollar-volume field and retains its source labels.

At Monday's close, rank using data through the preceding Sunday. Trade at Monday's close and first
earn Tuesday's close-to-close return, preserving the original execution delay. Monday price availability
is checked at execution. Long the top quintile and short the bottom quintile, each 50% of NAV;
weights drift between rebalances. Fewer than 10 eligible coins means cash. Size always requires caps.

Each chronological segment starts from cash with the same capital. Fees are 20 bps per absolute
traded notional, including segment-end liquidation. NAV changes only through marked position P&L
and fees. A missing held quote or insolvency stops that account and is explicitly reported; no
zero-return substitution, fabricated delisting proceeds, capital reset, or deleting failed dates.
Unheld missing quotes do not contaminate account returns. No borrow fees, margin liquidation model,
or slippage beyond the shared 20-bps assumption is available in this spot dataset.
### Equation-to-code map

| Calculation | Code | Meaning |
|---|---|---|
| Cross-sectional scores | `build_signals` | The nine rankings defined in the introduction |
| Top/bottom quintiles | `target_for_day` | Floor of eligible count / 5 coins on each side |
| Post-fee position sizing | `execute_target` | Solve fees and allocate +50% / -50% of after-fee NAV |
| Daily account | `run_account` | Mark positions, apply event exits and fees, then rebalance if scheduled |

The accounting identity is **ending NAV = starting NAV + daily position P&L - trading fees**.
Cash plus signed marked positions must equal NAV. The quantity of each token is implicit in the
position notional and observed prices; short sale proceeds do not create new capital.
Both selected universes require qualifying caps for every signal and use cap weights within each
leg. The core functions are retained from the template; the unused liquidity-policy helper is removed.


In [6]:
# vectorized signals and dollar-position accounting, no per-coin daily Python loop.
def build_signals(p, v, caps):
    v = v.where(np.isfinite(v) & v.gt(0))
    out = {'size':-caps, 'price':-p, 'max_price':-p.rolling(7,min_periods=7).max(),
           'dollar_volume':-v.rolling(7,min_periods=7).mean(),
           'volume_volatility':-v.rolling(7,min_periods=7).std(ddof=1)}
    for w in range(1,5):
        days = 7*w
        out[f'mom_{w}w'] = (p/p.shift(days)-1).where(p.rolling(days+1,min_periods=days+1).count().eq(days+1))
    return out

def target_for_day(i, dates, p, signal, caps, cap_weighted, require_cap, allowed=None):
    # exact preceding Sunday only; no cap interpolation or future signals.
    prior = i-1
    valid = np.isfinite(signal[prior]) & np.isfinite(p[i]) & (p[i]>0)
    if allowed is not None: valid &= allowed
    if require_cap:
        valid &= np.isfinite(caps[prior]) & (caps[prior]>=MIN_MARKET_CAP)
    indices = np.flatnonzero(valid)  # columns are sorted for deterministic tie breaking
    target = np.zeros(p.shape[1])
    if len(indices)>=MIN_ELIGIBLE and np.unique(signal[prior,indices]).size>=2:
        order = indices[np.argsort(signal[prior,indices],kind='stable')]
        k = len(order)//5
        for leg, sign in [(order[-k:],1), (order[:k],-1)]:
            leg_weights = caps[prior,leg] if cap_weighted else np.ones(k)
            target[leg] = sign*.5*leg_weights/leg_weights.sum()
    return target, len(indices)

def execute_target(nav, positions, target, cost=COST_RATE):
    # Solve fee = rate * dollars traded, using AFTER-fee NAV for target allocations.
    fee = 0.0
    for _ in range(100):
        next_fee = cost*np.abs(target*(nav-fee)-positions).sum()
        if abs(next_fee-fee)<1e-10*max(1,nav):
            fee=next_fee
            break
        fee=next_fee
    if fee>=nav or not np.isfinite(fee): raise ValueError('Trading costs exhaust NAV')
    new_positions=target*(nav-fee)
    traded=np.abs(new_positions-positions).sum()
    if not np.isclose(fee,cost*traded,atol=1e-7): raise AssertionError('Fee identity')
    return nav-fee, new_positions, fee, traded

def run_account(p_frame, signals, caps_frame, key, dates, cap_weighted, require_cap, cost=COST_RATE, policy=None, turnover_frame=None):
    p=p_frame.to_numpy(); sig=signals[key].to_numpy(); caps=caps_frame.to_numpy()
    positions=np.zeros(p.shape[1]); nav=INITIAL_CAPITAL
    blocked = policy['blocked'].to_numpy() if policy is not None else np.zeros_like(p,dtype=bool)
    can_execute = policy['can_execute'].to_numpy() if policy is not None else np.ones_like(p,dtype=bool)
    turnover = turnover_frame.to_numpy() if turnover_frame is not None else np.ones_like(p)
    # optional verified observations and pre-halt execution evidence, aligned exactly.
    def policy_array(name, default):
        if policy is None or name not in policy: return np.full(p.shape, default)
        frame = policy[name]
        if not frame.index.equals(p_frame.index) or not frame.columns.equals(p_frame.columns):
            raise ValueError('Event evidence must align with the price panel')
        return frame.to_numpy()
    entry_eligible = policy_array('entry_eligible', True).astype(bool)
    verified = policy_array('verified_extreme', False).astype(bool)
    exit_prices = policy_array('intraday_exit_price', np.nan)
    exit_volumes = policy_array('intraday_exit_volume', np.nan)
    exit_times = policy_array('intraday_exit_time', np.datetime64('NaT','ns'))
    rows=[]; selections=[]; failure=None
    for date in dates:
        i=p_frame.index.get_loc(date); start_nav=nav
        held=positions!=0
        # Mark only an affected held instrument at its verified pre-halt exit, not the partial daily close.
        intraday = held & blocked[i] & ~can_execute[i] & np.isfinite(exit_prices[i]) & (exit_prices[i]>0)
        executable_today = can_execute[i] | intraday
        mark = np.where(intraday, exit_prices[i], p[i])
        trade_volume = np.where(intraday, exit_volumes[i], turnover[i])
        if (held & ~executable_today).any():
            failure={'date':str(date.date()),'reason':'Held position reached known halt; intraday or settlement evidence required',
                     'assets':','.join(p_frame.columns[held & ~executable_today])}; break
        if held.any() and not (np.isfinite(mark[held]) & (mark[held]>0) & np.isfinite(p[i-1,held]) & (p[i-1,held]>0)).all():
            failure={'date':str(date.date()),'reason':'Missing held quote; NAV cannot be marked',
                     'assets':','.join(p_frame.columns[held & (~np.isfinite(mark) | (mark<=0) | ~np.isfinite(p[i-1]))])}
            break
        r=np.zeros(p.shape[1]); r[held]=mark[held]/p[i-1,held]-1
        # unexplained 10x moves need split/event review, never clipping or retrospective filtering.
        suspect=held & ((r>=9) | (r<=-.9)) & ~verified[i]
        if suspect.any():
            failure={'date':str(date.date()),'reason':'Extreme observed move requires event/data review',
                     'assets':','.join(p_frame.columns[suspect])}; break
        reviewed_move_pnl = float((positions*r)[held & verified[i]].sum())
        intraday_exit_pnl = float((positions*r)[intraday].sum())
        pnl=float(positions @ r); positions=positions*(1+r); nav+=pnl
        if not np.isfinite(nav):
            failure={'date':str(date.date()),'reason':'Non-finite NAV requires data review','assets':''}; break
        if nav<=0:
            failure={'date':str(date.date()),'reason':'Insolvent account; stopped','assets':''}; break
        fee=traded=event_fee=event_notional=0.0; eligible=np.nan
        # Exit only announced events, using today's actual executable close and today's costs.
        closing = (positions!=0) & blocked[i]
        if closing.any():
            if not (executable_today[closing] & np.isfinite(trade_volume[closing]) & (trade_volume[closing]>0)).all():
                failure={'date':str(date.date()),'reason':'Announced exit lacks executable positive-turnover quote',
                         'assets':','.join(p_frame.columns[closing])}; break
            # bound the minute-fill assumption; do not force an oversized fill.
            oversized = intraday & (np.abs(positions) > .01*trade_volume)
            if oversized.any():
                failure={'date':str(date.date()),'reason':'Intraday exit exceeds 1% of observed minute turnover',
                         'assets':','.join(p_frame.columns[oversized])}; break
            event_notional=float(np.abs(positions[closing]).sum()); event_fee=cost*event_notional
            for j in np.flatnonzero(closing):
                selections.append({'date':date,'asset':p_frame.columns[j],'target_weight':0.0,
                                   'phase':'intraday_event_exit' if intraday[j] else 'announced_event_exit',
                                   'trade_notional':abs(positions[j]), 'execution_price':mark[j],
                                   'execution_time':exit_times[i,j] if intraday[j] else date+pd.Timedelta(days=1),
                                   'execution_basis':'verified minute close' if intraday[j] else 'daily close'})
            positions[closing]=0; nav-=event_fee; fee+=event_fee; traded+=event_notional
            if nav<=0:
                failure={'date':str(date.date()),'reason':'Event exit costs exhaust NAV','assets':''}; break
        # No opening new trades on the final date immediately before liquidation.
        if date.weekday()==0 and date!=dates[-1]:
            allowed=(~blocked[i] & can_execute[i] & entry_eligible[i] & np.isfinite(turnover[i]) & (turnover[i]>0))
            target,eligible=target_for_day(i,p_frame.index,p,sig,caps,cap_weighted,require_cap,allowed)
            needs_trade = np.abs(target*nav-positions)>1e-8
            executable = can_execute[i] & np.isfinite(turnover[i]) & (turnover[i]>0)
            if (needs_trade & ~executable).any():
                failure={'date':str(date.date()),'reason':'Rebalance lacks executable positive-turnover quote',
                         'assets':','.join(p_frame.columns[needs_trade & ~executable])}; break
            try:
                nav,positions,reb_fee,reb_traded=execute_target(nav,positions,target,cost)
                fee+=reb_fee; traded+=reb_traded
            except ValueError as exc:
                failure={'date':str(date.date()),'reason':str(exc),'assets':''}; break
            for j in np.flatnonzero(target):
                selections.append({'date':date,'asset':p_frame.columns[j],'target_weight':target[j],'phase':'weekly_rebalance'})
        if date==dates[-1]:
            closing = positions!=0
            if not (can_execute[i,closing] & np.isfinite(turnover[i,closing]) & (turnover[i,closing]>0)).all():
                failure={'date':str(date.date()),'reason':'Final liquidation lacks executable positive-turnover quote',
                         'assets':','.join(p_frame.columns[closing])}; break
            end_traded=float(np.abs(positions).sum()); end_fee=cost*end_traded
            traded+=end_traded; fee+=end_fee; nav-=end_fee; positions[:]=0
        if nav<=0:
            failure={'date':str(date.date()),'reason':'Liquidation costs exhaust NAV','assets':''}; break
        if not np.isclose(nav,start_nav+pnl-fee,atol=1e-7): raise AssertionError('Capital conservation')
        rows.append({'date':date,'nav':nav,'cash':nav-positions.sum(),'positions_net':positions.sum(),
                     'gross_exposure':np.abs(positions).sum()/nav,'pnl':pnl,'fee':fee,
                     'traded_notional':traded,'turnover':traded/start_nav,'return':nav/start_nav-1,
                     'event_exit_notional':event_notional,'event_exit_fee':event_fee,
                     'reviewed_move_pnl':reviewed_move_pnl,'intraday_exit_pnl':intraday_exit_pnl,
                     'eligible_at_rebalance':eligible})
    frame=pd.DataFrame(rows)
    if len(frame): frame=frame.set_index('date')
    if failure:
        # distinguish an incomplete calculation from economic insolvency.
        reason = failure['reason'].lower()
        if 'insolvent' in reason or 'exhaust' in reason:
            failure['status'] = 'insolvent'
        elif any(s in reason for s in ['known halt','executable','liquidation lacks','intraday exit']):
            failure['status'] = 'execution_unavailable'
        else:
            failure['status'] = 'needs_data_review'
        failure['last_valid_nav'] = float(frame.nav.iloc[-1]) if len(frame) else INITIAL_CAPITAL
    return frame,pd.DataFrame(selections),failure

## 6. Accounting, timing and event tests
<!-- CHANGED: synthetic tests cover missing unheld quotes, unavailable held exits, fee identities,
short insolvency, target budgets and independence from future observations. -->
Retain fee, NAV, timing, missing-price, insolvency, identity and reviewed-event tests. The removed
liquidity experiment has no remaining configuration or dedicated tests in this notebook.


In [7]:
# Deterministic checks before real-data runs.
test_dates=pd.date_range('2024-01-01',periods=45)
test_columns=[f'A{i:02}' for i in range(10)]
tp=pd.DataFrame(100.0,index=test_dates,columns=test_columns)
tv=pd.DataFrame(1000.0,index=test_dates,columns=test_columns)
tc=pd.DataFrame(np.tile(np.arange(1,11)*1e6,(45,1)),index=test_dates,columns=test_columns)
ts=build_signals(tp,tv,tc)
t,count=target_for_day(7,test_dates,tp.to_numpy(),ts['size'].to_numpy(),tc.to_numpy(),True,True)
assert count==10 and np.isclose(t[t>0].sum(),.5) and np.isclose(t[t<0].sum(),-.5)
nav,pos,fee,trade=execute_target(100000,np.zeros(10),t)
assert np.isclose(nav+fee,100000) and np.isclose(fee,COST_RATE*trade)
assert np.isclose(pos.sum(),0)
future=tp.copy(); future.iloc[20:]*=2
future_s=build_signals(future,tv,tc)
for k in STRATEGIES: pd.testing.assert_frame_equal(ts[k].iloc[:20],future_s[k].iloc[:20])
f,_,err=run_account(tp,ts,tc,'size',test_dates[1:],True,True,cost=0)
assert err is None and np.allclose(f.nav,INITIAL_CAPITAL)
f,_,err=run_account(tp,ts,tc,'size',test_dates[1:],True,True)
assert err is None and np.isclose(f.nav.iloc[-1],INITIAL_CAPITAL-f.fee.sum())
assert f.loc['2024-01-07','gross_exposure']==0 and f.loc['2024-01-08','gross_exposure']>0
# Unheld missing prices must not cause 0 * NaN contamination.
gap=tp.copy(); gap.iloc[8,4]=np.nan
f,_,err=run_account(gap,ts,tc,'size',test_dates[1:],True,True)
assert err is None
# A held missing price must stop; no invented proceeds or complete-case deletion.
gap=tp.copy(); gap.iloc[8,0]=np.nan
f,_,err=run_account(gap,ts,tc,'size',test_dates[1:],True,True)
assert err and err['date']=='2024-01-09' and 'Missing held' in err['reason']
crash=tp.copy(); crash.iloc[8:,8:]=900
_,_,err=run_account(crash,ts,tc,'size',test_dates[1:],True,True)
assert err and 'Insolvent' in err['reason']
missing_caps=tc.to_numpy().copy(); missing_caps[6]=np.nan
empty,n=target_for_day(7,test_dates,tp.to_numpy(),ts['price'].to_numpy(),missing_caps,True,True)
assert n==0 and np.all(empty==0)
print('PASS: budgets, fee accounting, conservation, execution delay, future independence, missing quotes, insolvency and exact-date caps.')
# event exits and split-unit tests, including short liabilities.
te=pd.DataFrame([{'symbol':'A00','announced_at':pd.Timestamp('2024-01-09 12:00'),
    'halt_at':pd.Timestamp('2024-01-11 03:00'),'resume_at':pd.Timestamp('2024-01-15 03:00'),
    'replacement':'A00','new_units_per_old':.001}])
pol=compile_event_policy(test_dates,test_columns,te)
assert not pol['blocked'].loc['2024-01-08','A00'] and pol['blocked'].loc['2024-01-09','A00']
assert not pol['can_execute'].loc['2024-01-11','A00']
splitp=tp.copy(); splitp.loc['2024-01-11':'2024-01-15','A00']=np.nan
splitp.loc['2024-01-16':,'A00']=100000
splits=build_signals(splitp.mask(pol['reset_signal']),tv,tc)
ef,et,err=run_account(splitp,splits,tc,'size',test_dates[1:],True,True,policy=pol,turnover_frame=tv)
assert err is None and ef.event_exit_notional.sum()>0
assert np.isclose(ef.nav.iloc[-1],INITIAL_CAPITAL-ef.fee.sum())
assert len(et.loc[et.phase.eq('announced_event_exit')])==1
for units in [1000000,-1000000]:
    assert np.isclose(converted_value(units,.01,.0001),units*.01)
# An unknown large jump is a review failure, not reported as proven economic insolvency.
bad=tp.copy(); bad.iloc[8:,9]=100000
_,_,err=run_account(bad,ts,tc,'size',test_dates[1:],True,True)
assert err and 'review' in err['reason']
# verify the exclusion filter without constructing a leveraged-token experiment.
assert product_audit.product_class.eq('leveraged_token').sum()==40 and 'JUPUSDT' not in leveraged_symbols
# Events announced in the future cannot change prior eligibility or signals.
future_events=te.copy()
future_events[['announced_at','halt_at','resume_at']]+=pd.Timedelta(days=100)
future_policy=compile_event_policy(test_dates,test_columns,future_events)
assert not future_policy['blocked'].to_numpy().any() and future_policy['can_execute'].to_numpy().all()
# an announcement and halt within one day cannot be exited at yesterday's close.
late=te.copy()
late['announced_at']=pd.Timestamp('2024-01-09 00:28')
late['halt_at']=pd.Timestamp('2024-01-09 00:50')
late_policy=compile_event_policy(test_dates,test_columns,late)
_,_,err=run_account(tp,ts,tc,'size',test_dates[1:],True,True,policy=late_policy,turnover_frame=tv)
assert err and err['date']=='2024-01-09' and 'known halt' in err['reason']
# Final liquidation cannot invent execution on a zero-turnover day.
zero_turnover=tv.copy(); zero_turnover.iloc[-1,:]=0
_,_,err=run_account(tp,ts,tc,'size',test_dates[1:],True,True,turnover_frame=zero_turnover)
assert err and 'liquidation lacks' in err['reason']
print('PASS: announcement timing, halt exclusion, split reset, paid event exits, value conservation and product taxonomy.')

# a verified economic crash credits shorts and debits longs without changing capital otherwise.
# Test both position signs with a real 99% price decline, then a pre-halt minute exit.
for affected, expected_sign in [('A00', -1), ('A09', 1)]:
    reviewed_p = tp.copy()
    reviewed_p.loc['2024-01-09':, affected] = 1.0
    reviewed_s = build_signals(reviewed_p,tv,tc)
    reviewed_policy = compile_event_policy(test_dates,test_columns,te.iloc[0:0])
    reviewed_policy['verified_extreme'] = pd.DataFrame(False,index=test_dates,columns=test_columns)
    reviewed_policy['verified_extreme'].at[pd.Timestamp('2024-01-09'),affected] = True
    ledger,_,stop = run_account(reviewed_p,reviewed_s,tc,'size',test_dates[1:12],True,True,cost=0,policy=reviewed_policy)
    assert stop is None and expected_sign*ledger.at[pd.Timestamp('2024-01-09'),'reviewed_move_pnl']>0
    np.testing.assert_allclose(ledger.nav,INITIAL_CAPITAL+ledger.pnl.cumsum()-ledger.fee.cumsum())
    event = te.copy()
    event['symbol'] = affected
    event['announced_at'] = pd.Timestamp('2024-01-10 00:28')
    event['halt_at'] = pd.Timestamp('2024-01-10 00:40')
    event['resume_at'] = pd.NaT
    event['replacement'] = ''
    exit_policy = compile_event_policy(test_dates,test_columns,event)
    exit_policy['verified_extreme'] = reviewed_policy['verified_extreme']
    for key in ['intraday_exit_price','intraday_exit_volume']:
        exit_policy[key] = pd.DataFrame(np.nan,index=test_dates,columns=test_columns)
    exit_policy['intraday_exit_time'] = pd.DataFrame(pd.NaT,index=test_dates,columns=test_columns)
    date = pd.Timestamp('2024-01-10')
    exit_policy['intraday_exit_price'].at[date,affected] = .8
    exit_policy['intraday_exit_volume'].at[date,affected] = 1000000
    exit_policy['intraday_exit_time'].at[date,affected] = pd.Timestamp('2024-01-10 00:29:59.999')
    # An unusable or wildly different post-halt daily close must not affect the minute fill.
    reviewed_p.loc[date:,affected] = 1000000
    ledger,orders,stop = run_account(reviewed_p,reviewed_s,tc,'size',test_dates[1:12],True,True,policy=exit_policy,turnover_frame=tv)
    assert stop is None and expected_sign*ledger.at[date,'intraday_exit_pnl']>0
    fills = orders.loc[orders.phase.eq('intraday_event_exit')]
    assert len(fills)==1 and fills.execution_price.iloc[0]==.8
    assert fills.execution_time.iloc[0] < event.halt_at.iloc[0]
    assert ledger.at[date,'event_exit_fee']>0
    np.testing.assert_allclose(ledger.nav,INITIAL_CAPITAL+ledger.pnl.cumsum()-ledger.fee.cumsum())
    # No verified minute means no invented proceeds. Excessive size is also not force-filled.
    no_fill = dict(exit_policy); no_fill['intraday_exit_price']=exit_policy['intraday_exit_price']*np.nan
    _,_,stop = run_account(reviewed_p,reviewed_s,tc,'size',test_dates[1:12],True,True,policy=no_fill)
    assert stop['status']=='execution_unavailable'
    too_large = dict(exit_policy); too_large['intraday_exit_volume']=exit_policy['intraday_exit_volume']*0+1
    _,_,stop = run_account(reviewed_p,reviewed_s,tc,'size',test_dates[1:12],True,True,policy=too_large)
    assert stop['status']=='execution_unavailable'
_,_,stop = run_account(bad,ts,tc,'size',test_dates[1:],True,True)
assert stop['status']=='needs_data_review'
_,_,stop = run_account(crash,ts,tc,'size',test_dates[1:],True,True)
assert stop['status']=='insolvent'
assert binance_prices.loc[binance_prices.index<LUNA_NEW_START,'LUNAUSDT'].isna().all()
assert binance_prices.loc[binance_prices.index>=pd.Timestamp('2022-05-14'),LUNA_OLD].isna().all()
assert event_policy['blocked'].loc['2022-06-01':,LUNA_OLD].all()
print('PASS: reviewed long/short P&L, intraday exits, fee/NAV identities, unavailable/oversized fills, separate LUNA epochs and distinct stop statuses.')


# timestamp precision must not reject valid evidence or accept missing/corrupt bars.
minute_test_day = '2022-05-10'
minute_test_grid = pd.date_range(minute_test_day, periods=1440, freq='min')
for unit in ['ns','us','ms']:
    validate_complete_minute_day(pd.Series(minute_test_grid.as_unit(unit)), minute_test_day)
for invalid_minutes in [
    minute_test_grid.delete(700),
    minute_test_grid[:700].append(minute_test_grid[699:1439]),
    minute_test_grid + pd.Timedelta(seconds=1),
    minute_test_grid[::-1],
]:
    try:
        validate_complete_minute_day(invalid_minutes, minute_test_day)
    except ValueError:
        pass
    else:
        raise AssertionError('Corrupt or incomplete minute evidence was accepted')
print('Minute evidence precision, missing-bar, duplicate, offset and ordering checks passed.')

# regression checks for the four newly reviewed event gaps.
# Verify observed executable exits, no pre-announcement blocking, and signal reset at halt.
for reviewed_symbol, reviewed_day in [('AUDUSDT','2023-05-26'),('TVKUSDT','2023-11-17'),
                                     ('TOMOUSDT','2023-11-06'),('STRAXUSDT','2024-03-08')]:
    reviewed_day = pd.Timestamp(reviewed_day)
    reviewed_event = events.loc[events.symbol.eq(reviewed_symbol)
        & events.announced_at.dt.normalize().eq(reviewed_day)].iloc[0]
    assert reviewed_event.announced_at < reviewed_day + pd.Timedelta(days=1) < reviewed_event.halt_at
    assert binance_prices.at[reviewed_day, reviewed_symbol] > 0
    assert binance_volume.at[reviewed_day, reviewed_symbol] > 0
    assert not event_policy['blocked'].at[reviewed_day-pd.Timedelta(days=1),reviewed_symbol]
    assert event_policy['blocked'].at[reviewed_day,reviewed_symbol]
    assert event_policy['can_execute'].at[reviewed_day,reviewed_symbol]
    assert event_policy['reset_signal'].at[reviewed_event.halt_at.normalize(),reviewed_symbol]
assert signal_prices.loc['2024-03-20':'2024-03-28','STRAXUSDT'].isna().all()
assert not event_policy['blocked'].at[pd.Timestamp('2024-03-29'),'STRAXUSDT']
print('Reviewed AUD, TVK, TOMO and STRAX event checks passed.')

# UTK uses an observed post-announcement/pre-halt fill, with no future daily mark.
assert len(utk_minutes)==840 and utk_exit.closed < utk_event.halt_at
assert utk_exit.opened == utk_event.announced_at+pd.Timedelta(minutes=1)
assert np.isclose(float(utk_exit['close']),.01433)
assert np.isclose(float(utk_exit.quote_volume),223043.10102)
if utk_exit_day in calendar:
    assert event_policy['blocked'].at[utk_exit_day,'UTKUSDT']
    assert not event_policy['can_execute'].at[utk_exit_day,'UTKUSDT']
    assert event_policy['intraday_exit_price'].at[utk_exit_day,'UTKUSDT']==float(utk_exit['close'])
    assert pd.isna(event_policy['intraday_exit_price'].at[utk_exit_day-pd.Timedelta(days=1),'UTKUSDT'])
print('UTK checksum, timestamp units, partial-day coverage and exit timing checks passed.')


PASS: budgets, fee accounting, conservation, execution delay, future independence, missing quotes, insolvency and exact-date caps.
PASS: announcement timing, halt exclusion, split reset, paid event exits, value conservation and product taxonomy.
PASS: reviewed long/short P&L, intraday exits, fee/NAV identities, unavailable/oversized fills, separate LUNA epochs and distinct stop statuses.
Minute evidence precision, missing-bar, duplicate, offset and ordering checks passed.
Reviewed AUD, TVK, TOMO and STRAX event checks passed.
UTK checksum, timestamp units, partial-day coverage and exit timing checks passed.


## 7. Execute the two universes in backtest and validation

Run nine signals in each of `cmc_30_cap` and `binance_matched_cap`, gross and net of costs.
With both periods selected, this produces 72 account outcomes: 2 universes x 9 signals x 2 periods
x 2 cost settings. No other universe is evaluated. Preflight broad weekly cap coverage for Binance;
retain partial ledgers and explicit failure reports when data or required exits are unsupported.
Complete accounts receive total return, CAGR, Sharpe, drawdown, turnover and fee metrics.


In [8]:
# precompute signals once per source and reuse across experiments.
period_dates=calendar[1:]
# explicit inclusive windows, independent of downloaded history length.
PERIOD_DATES = {
    'Backtest': pd.date_range(BACKTEST_START, BACKTEST_END, freq='D', name='date'),
    'Validation': pd.date_range(VALIDATION_START, VALIDATION_END, freq='D', name='date'),
}
if RUN_SCOPE == 'backtest':
    PERIOD_DATES = {'Backtest': PERIOD_DATES['Backtest']}
    print('Validation NOT RUN: switch RUN_SCOPE to both after updating the source snapshots.')
combined_dates = pd.DatetimeIndex([],name='date')
for selected_dates in PERIOD_DATES.values(): combined_dates = combined_dates.append(selected_dates)
assert combined_dates.equals(period_dates), 'Selected periods must exactly cover the loaded return calendar'
display(pd.DataFrame([{'Period':k,'Start':v[0],'End':v[-1],'Days':len(v)} for k,v in PERIOD_DATES.items()]))
baseline_signals=build_signals(baseline_prices,baseline_volume,baseline_caps)
binance_signals=build_signals(signal_prices,signal_volume,binance_caps)
ordinary_prices=binance_prices[ordinary]
ordinary_caps=binance_caps[ordinary]
ordinary_signals={k:v[ordinary] for k,v in binance_signals.items()}
experiments = {
    'cmc_30_cap':(baseline_prices,baseline_signals,baseline_caps,True,True,None,None),
    'binance_matched_cap':(ordinary_prices,ordinary_signals,ordinary_caps,True,True,select_policy(ordinary),binance_volume[ordinary]),
}
assert baseline_experiment == 'cmc_30_cap'
assert set(experiments) == {'cmc_30_cap','binance_matched_cap'}

def metrics(frame, benchmark):
    r=frame['return']; wealth=frame.nav/INITIAL_CAPITAL
    drawdown=wealth/wealth.cummax().clip(lower=1)-1
    sd=r.std(ddof=1)
    return {**btc_alpha_metrics(r, benchmark), 'Total return (%)':100*(wealth.iloc[-1]-1),
            'CAGR (%)':100*(wealth.iloc[-1]**(365/len(frame))-1),
            'Sharpe':np.sqrt(SHARPE_PERIODS_PER_YEAR)*r.mean()/sd if sd>1e-15 else np.nan, 'Sharpe (365)':np.sqrt(365)*r.mean()/sd if sd>1e-15 else np.nan,
            'Max drawdown (%)':100*drawdown.min(),'Final capital':frame.nav.iloc[-1],
            'Fees paid':frame.fee.sum(),'Mean daily turnover':frame.turnover.mean(),
            'Mean eligible':frame.eligible_at_rebalance.mean()}
def missing_weekly_cap_dates(dates, available):
    rebalances = dates[(dates.weekday == 0) & (dates != dates[-1])]
    return (rebalances - pd.Timedelta(days=1)).difference(available)

results={}; constituents={}; comparison_rows=[]; failures=[]
for experiment,(p,s,c,cap_weights,require_caps,policy,turnover) in experiments.items():
    print('Running',experiment,flush=True)
    for period,dates in PERIOD_DATES.items():
        for key in STRATEGIES:
            for cost_label,cost in [('gross',0.0),('net',COST_RATE)]:
                broad_cap_required = experiment != baseline_experiment and (cap_weights or require_caps or key=='size')
                missing_caps = missing_weekly_cap_dates(dates,weekly_available_dates) if broad_cap_required else pd.DatetimeIndex([])
                if len(missing_caps):
                    frame,selected_coins = pd.DataFrame(),pd.DataFrame()
                    failure = {'status':'needs_data_review','date':str(missing_caps[0].date()),
                        'reason':'Missing broad weekly CMC snapshot(s): '+', '.join(str(d.date()) for d in missing_caps),
                        'assets':'broad CMC universe','last_valid_nav':INITIAL_CAPITAL}
                else:
                    frame,selected_coins,failure=run_account(p,s,c,key,dates,cap_weights,require_caps or key=='size',cost,policy,turnover)
                tag=(experiment,period,key,cost_label)
                results[tag]=frame; constituents[tag]=selected_coins
                row=dict(zip(['Experiment','Period','Strategy','Costs'],tag))
                row.update({'Status':failure['status'] if failure else 'complete','Observed days':len(frame)})
                if failure:
                    failures.append({**row,**failure})
                else:
                    assert frame.index.equals(dates), 'Incomplete successful account'
                    benchmark = (baseline_prices['BTC-USD'] if experiment == 'cmc_30_cap' else binance_prices['BTCUSDT']).pct_change(fill_method=None)
                    row.update(metrics(frame, benchmark))
                comparison_rows.append(row)
comparison=pd.DataFrame(comparison_rows)
# Retain a readable failure-report schema when every account completes.
failure_table=pd.DataFrame(failures,columns=['Experiment','Period','Strategy','Costs','Status','Observed days','date','reason','assets','status','last_valid_nav'])
net_comparison=comparison.loc[comparison.Costs.eq('net')].copy()
net_comparison['Sharpe rank']=net_comparison.groupby(['Experiment','Period']).Sharpe.rank(ascending=False,method='min')
# Keep inference visible instead of letting a wide DataFrame hide the CI columns.
headline_columns = ['Experiment', 'Period', 'Strategy', 'Total return (%)', 'Sharpe', 'Sharpe (365)',
                    'Alpha vs BTC (%)', 'Alpha HAC t', 'Alpha 95% CI low (%)', 'Alpha 95% CI high (%)']
with pd.option_context('display.max_columns', None):
    display(net_comparison[headline_columns])
display(net_comparison)
display(failure_table)
status_summary=comparison.groupby(['Experiment','Period','Costs','Status']).size().rename('Accounts').reset_index()
display(status_summary)
if len(failure_table):
    print('Failure causes (gross and net accounts counted separately):')
    display(failure_table.groupby(['Experiment','reason']).size().rename('Accounts'))
print('Statuses: complete, needs_data_review, execution_unavailable, insolvent. Only complete accounts receive full-period metrics.')

,Period,Start,End,Days
0,Backtest,2021-01-02,2024-12-31,1460
1,Validation,2025-01-01,2026-09-17,625


Running cmc_30_cap


Running binance_matched_cap


,Experiment,Period,Strategy,Total return (%),Sharpe,Sharpe (365),Alpha vs BTC (%),Alpha HAC t,Alpha 95% CI low (%),Alpha 95% CI high (%)
1,cmc_30_cap,Backtest,size,425.303058,1.154547,1.389498,43.364944,1.978622,0.373224,86.356663
3,cmc_30_cap,Backtest,price,267.889318,0.855505,1.029601,39.426129,1.615485,-8.446807,87.299065
5,cmc_30_cap,Backtest,max_price,277.933049,0.872288,1.049799,39.882108,1.636672,-7.917608,87.681823
7,cmc_30_cap,Backtest,mom_1w,-3.975994,0.133585,0.160769,8.377913,0.490869,-25.101601,41.857426
9,cmc_30_cap,Backtest,mom_2w,-22.736434,0.003060,0.003683,1.435750,0.082162,-32.842376,35.713876
11,cmc_30_cap,Backtest,mom_3w,-8.681734,0.102317,0.123138,4.324508,0.254007,-29.071961,37.720976
13,cmc_30_cap,Backtest,mom_4w,-44.110688,-0.227596,-0.273912,-8.821859,-0.562791,-39.570209,21.926492
15,cmc_30_cap,Backtest,dollar_volume,222.487789,0.910813,1.096164,32.620783,1.525423,-9.327373,74.568939
17,cmc_30_cap,Backtest,volume_volatility,288.531256,1.047491,1.260656,36.520389,1.804480,-3.179766,76.220545
19,cmc_30_cap,Validation,size,-44.746817,-1.018981,-1.226343,-30.982105,-1.817327,-64.460946,2.496735


,Experiment,Period,Strategy,Costs,Status,Observed days,Alpha vs BTC (%),Beta vs BTC,Alpha HAC p,Alpha HAC t,...,Total return (%),CAGR (%),Sharpe,Sharpe (365),Max drawdown (%),Final capital,Fees paid,Mean daily turnover,Mean eligible,Sharpe rank
1,cmc_30_cap,Backtest,size,net,complete,1460,43.364944,0.077323,0.048047,1.978622,...,425.303058,51.391845,1.154547,1.389498,-25.933575,525303.057909,29804.196177,0.024931,29.990431,1.0
3,cmc_30_cap,Backtest,price,net,complete,1460,39.426129,0.003772,0.106422,1.615485,...,267.889318,38.493469,0.855505,1.029601,-45.731463,367889.317801,15465.899709,0.017760,29.990431,5.0
5,cmc_30_cap,Backtest,max_price,net,complete,1460,39.882108,0.007060,0.101915,1.636672,...,277.933049,39.429194,0.872288,1.049799,-47.934213,377933.048986,15904.723765,0.017461,29.846890,4.0
7,cmc_30_cap,Backtest,mom_1w,net,complete,1460,8.377913,-0.045583,0.623593,0.490869,...,-3.975994,-1.009172,0.133585,0.160769,-62.112401,96024.006220,86028.047874,0.250961,29.846890,6.0
9,cmc_30_cap,Backtest,mom_2w,net,complete,1460,1.435750,-0.027101,0.934529,0.082162,...,-22.736434,-6.245162,0.003060,0.003683,-71.338605,77263.565762,49806.277660,0.193423,29.703349,8.0
11,cmc_30_cap,Backtest,mom_3w,net,complete,1460,4.324508,0.005632,0.799526,0.254007,...,-8.681734,-2.244902,0.102317,0.123138,-50.410483,91318.265686,52890.047638,0.174111,29.559809,7.0
13,cmc_30_cap,Backtest,mom_4w,net,complete,1460,-8.821859,-0.005791,0.573664,-0.562791,...,-44.110688,-13.536623,-0.227596,-0.273912,-63.634830,55889.312249,30840.281131,0.149419,29.416268,9.0
15,cmc_30_cap,Backtest,dollar_volume,net,complete,1460,32.620783,0.028289,0.127371,1.525423,...,222.487789,34.007257,0.910813,1.096164,-51.162590,322487.789302,47419.525283,0.043968,29.846890,3.0
17,cmc_30_cap,Backtest,volume_volatility,net,complete,1460,36.520389,0.041263,0.071362,1.804480,...,288.531256,40.396568,1.047491,1.260656,-32.913655,388531.255511,62547.873329,0.069624,29.846890,2.0
19,cmc_30_cap,Validation,size,net,complete,625,-30.982105,0.166880,0.069647,-1.817327,...,-44.746817,-29.280910,-1.018981,-1.226343,-48.945458,55253.183469,1596.415212,0.019275,29.235955,6.0


,Experiment,Period,Strategy,Costs,Status,Observed days,date,reason,assets,status,last_valid_nav


,Experiment,Period,Costs,Status,Accounts
0,binance_matched_cap,Backtest,gross,complete,9
1,binance_matched_cap,Backtest,net,complete,9
2,binance_matched_cap,Validation,gross,complete,9
3,binance_matched_cap,Validation,net,complete,9
4,cmc_30_cap,Backtest,gross,complete,9
5,cmc_30_cap,Backtest,net,complete,9
6,cmc_30_cap,Validation,gross,complete,9
7,cmc_30_cap,Validation,net,complete,9


Statuses: complete, needs_data_review, execution_unavailable, insolvent. Only complete accounts receive full-period metrics.


## 8. Equity, Bitcoin benchmarks and reproducible outputs

Export results, failure reports, ledgers, constituents, source hashes and audit evidence for the two
selected experiments to this notebook's separate output directory. Draw backtest, validation and
linked combined growth graphs for each universe. A strategy appears only if it completed both
periods; negative performance remains visible. If a strategy fails validation, its complete backtest
still remains in the results table, even though the final graphs exclude it.

CMC Bitcoin buy and hold uses BTC-USD and USD capital; Binance uses BTCUSDT and USDT capital.
Both use the same initial capital and fee rate on purchase and final sale. Combined strategy curves
scale validation NAV by ending backtest NAV, retaining the existing liquidation/restart at the
boundary without injecting capital. Bitcoin stays invested across the boundary. This linked graph
is not a new continuous-position strategy simulation. Currency/source differences remain.


In [9]:
from html import escape
spec={'run_scope':RUN_SCOPE,'executed_periods':list(PERIOD_DATES),'loaded_end':DATA_END,'source_run':SOURCE_RUN,'sample':[SAMPLE_START,SAMPLE_END],'cost_bps':COST_BPS,
      'initial_capital':INITIAL_CAPITAL,
      'periods':{'Backtest':[BACKTEST_START,BACKTEST_END], 'Validation':[VALIDATION_START,VALIDATION_END]},
     'baseline_experiment':baseline_experiment,
      'missing_weekly_cap_dates':[str(d.date()) for d in missing_weekly_dates],
      'expanded_cap_manifest':cap_expansion_manifest, 'cap_price_ratio_limit':CAP_PRICE_RATIO_LIMIT,
      'text_checksum_verification':text_checksum_verification,
      'minimum_coins':MIN_ELIGIBLE,
      'evaluated_experiments':list(experiments), 'liquidity_filter_applied':False,
      'minimum_cap':MIN_MARKET_CAP,'inputs':source_hashes,'partitions':partition_hashes,
      'notebook_source_sha256':hashlib.sha256(json.dumps(
          [(c['cell_type'],c['source']) for c in json.loads((NOTEBOOK_DIR/NOTEBOOK_NAME).read_text(encoding='utf8'))['cells']],
          ensure_ascii=False,sort_keys=True).encode()).hexdigest(), 'engine_version':6,
      'event_policy':'first daily close after announcement; reviewed LUNA uses first minute bar after 1-minute reaction delay before halt',
      'luna_identity_source':LUNA_IDENTITY_SOURCE, 'intraday_participation_limit':0.01,
      'utk_exit_source':utk_manifest, 'intraday_reaction_delay_minutes':1,
      'product_source':PRODUCT_SOURCE}
OUTPUT_DIR=NOTEBOOK_DIR/'output/Common_Risk_Factors_Liu_et_al'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
comparison.to_csv(OUTPUT_DIR/'comparison.csv',index=False)
net_comparison.to_csv(OUTPUT_DIR/'net_rankings.csv',index=False)
failure_table.to_csv(OUTPUT_DIR/'failures.csv',index=False)
status_summary.to_csv(OUTPUT_DIR/'status_summary.csv',index=False)
events.to_csv(OUTPUT_DIR/'verified_market_events.csv',index=False)
cmc_source_audit.to_csv(OUTPUT_DIR/'cmc_30_provenance.csv',index=False)
cmc_cap_issues.to_csv(OUTPUT_DIR/'cmc_30_cap_issues.csv',index=False)
# export coverage and every accepted/rejected coin-week for reproducibility.
cap_candidates.to_csv(OUTPUT_DIR/'cap_identity_candidates.csv',index=False)
cap_match_audit.to_csv(OUTPUT_DIR/'cap_match_audit.csv',index=False)
# export the exact evidence and distinct LUNA identities alongside performance.
luna_reviews.to_csv(OUTPUT_DIR/'luna_verified_moves.csv',index=False)
luna_exit_evidence.to_csv(OUTPUT_DIR/'luna_exit_evidence.csv',index=False)
# export the independently verified UTK execution evidence.
utk_exit_evidence.to_csv(OUTPUT_DIR/'utk_exit_evidence.csv',index=False)
luna_identity_audit.to_csv(OUTPUT_DIR/'luna_identity_audit.csv',index=False)
repair_audit.to_csv(OUTPUT_DIR/'supplemental_candle_audit.csv',index=False)
product_audit.to_csv(OUTPUT_DIR/'product_audit.csv',index=False)
pair_audit.to_csv(OUTPUT_DIR/'pair_coverage.csv',index=False)
identity.to_csv(OUTPUT_DIR/'identity_audit.csv')
# Make accepted line-ending differences visible in exported verification evidence.
pd.DataFrame.from_dict(text_checksum_verification,orient='index').rename_axis('file').to_csv(OUTPUT_DIR/'text_checksum_verification.csv')
input_summary.to_csv(OUTPUT_DIR/'input_summary.csv',index=False)
coverage=pd.DataFrame({'price_days':binance_prices.notna().sum(),
                       'valid_sunday_caps':binance_caps.notna().sum(),
                       'first_price':binance_prices.apply(pd.Series.first_valid_index),
                       'last_price':binance_prices.apply(pd.Series.last_valid_index)})
coverage.to_csv(OUTPUT_DIR/'asset_coverage.csv')
pd.DataFrame([{'date':d,'period':label} for label,ds in PERIOD_DATES.items() for d in ds]).to_csv(OUTPUT_DIR/'calendar.csv',index=False)
palette=['#176b87','#d08a35','#8555a6','#268779','#b14e64','#5e70b5','#8a753d','#c3652c','#626c78']
for tag,frame in results.items():
    stem='__'.join(tag)
    frame.to_csv(OUTPUT_DIR/(stem+'_ledger.csv'))
    if len(constituents[tag]): constituents[tag].to_csv(OUTPUT_DIR/(stem+'_constituents.csv'),index=False)
# show both the cap-weighted Binance matched universe and the original 30-coin CMC universe.
# Require completion in BOTH periods separately for each experiment/strategy pair.
FINAL_GRAPH_EXPERIMENTS = ['cmc_30_cap', 'binance_matched_cap']
final_graph_status = (net_comparison.loc[
    net_comparison.Experiment.isin(FINAL_GRAPH_EXPERIMENTS)
    & net_comparison.Costs.eq('net')
    & net_comparison.Period.isin(['Backtest','Validation'])
].pivot(index=['Experiment','Strategy'], columns='Period', values='Status')
 .reindex(columns=['Backtest','Validation']))
final_graph_pairs = set(final_graph_status.index[final_graph_status.eq('complete').all(axis=1)])
# Bitcoin uses the matching universe source and currency, with the same capital and fees.
# No interim Bitcoin trading; entry sizing includes the purchase fee.
def bitcoin_hold_nav(dates, bitcoin_prices):
    entry_day = dates[0] - pd.Timedelta(days=1)
    quotes = bitcoin_prices.reindex(pd.DatetimeIndex([entry_day]).append(dates))
    if not np.isfinite(quotes).all() or not quotes.gt(0).all():
        raise ValueError('Bitcoin needs every daily price, including the prior-day entry close.')
    units = INITIAL_CAPITAL / (float(quotes.iloc[0]) * (1 + COST_RATE))
    nav = units * quotes.iloc[1:]
    nav.iloc[-1] *= 1 - COST_RATE  # final liquidation only
    return nav.rename('bitcoin_hold')

def draw_equity_comparison(frame, title, stem, subtitle, boundary=None):
    # NAV exports use the currency stated in the subtitle (USD for CMC; USDT for Binance).
    frame.to_csv(OUTPUT_DIR / (stem + '_equity.csv'), index_label='date')
    width, height, left, right, top, bottom = 1200, 670, 85, 1150, 105, 460
    dates = frame.index
    values = frame / INITIAL_CAPITAL
    high = max(1.0, float(values.max().max())) * 1.04
    def x(date): return left + (date-dates[0])/(dates[-1]-dates[0])*(right-left)
    def y(value): return bottom - value/high*(bottom-top)
    colors = dict(zip(STRATEGIES, palette)); colors['bitcoin_hold'] = '#111827'
    labels = {**STRATEGIES, 'bitcoin_hold': 'Bitcoin buy and hold'}
    svg = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {width} {height}" role="img" aria-label="{escape(title)}" style="font-family:Arial,sans-serif;max-width:100%;height:auto">',
           f'<rect width="{width}" height="{height}" fill="white"/>',
           f'<text x="25" y="30" font-size="22" font-weight="bold">{escape(title)}</text>',
           f'<text x="25" y="55" font-size="13" fill="#475569">{escape(subtitle)}</text>',
           f'<text x="25" y="78" font-size="13" fill="#475569">Growth of {INITIAL_CAPITAL:,.0f} {graph_currency} | net of fees | strategies complete in both periods</text>']
    for tick in np.linspace(0, high, 6):
        svg.append(f'<path d="M{left} {y(tick):.2f} H{right}" stroke="#e2e8f0"/><text x="15" y="{y(tick)+4:.2f}" font-size="12">{tick:.2f}x</text>')
    for date in pd.date_range(dates[0], dates[-1], periods=5):
        svg.append(f'<text x="{x(date):.2f}" y="485" text-anchor="middle" font-size="12">{date:%Y-%m-%d}</text>')
    if boundary is not None:
        svg.append(f'<path d="M{x(boundary):.2f} {top} V{bottom}" stroke="#64748b" stroke-dasharray="5 4"/><text x="{x(boundary)+8:.2f}" y="120" font-size="12">Validation starts</text>')
    for j,key in enumerate(frame.columns):
        points = ' '.join(f'{x(date):.2f},{y(value):.2f}' for date,value in values[key].items())
        style = 'stroke-width="3" stroke-dasharray="8 4"' if key=='bitcoin_hold' else 'stroke-width="1.7"'
        svg.append(f'<polyline points="{points}" fill="none" stroke="{colors[key]}" {style}><title>{escape(labels[key])}</title></polyline>')
        lx,ly = 30+(j%3)*390, 530+(j//3)*29
        svg.append(f'<path d="M{lx} {ly-4} h24" stroke="{colors[key]}" {style}/><text x="{lx+32}" y="{ly}" font-size="12" fill="{colors[key]}">{escape(labels[key])}</text>')
    svg.append('</svg>')
    chart=''.join(svg)
    (OUTPUT_DIR/(stem+'.svg')).write_text(chart,encoding='utf8')
    display(HTML(chart))

# separate panels keep 18 strategy lines readable and avoid mixing USD/USDT NAV.
# Both universes remain together in the final Sharpe ranking graphs.
equity_by_universe = {}
for graph_experiment in FINAL_GRAPH_EXPERIMENTS:
    final_graph_strategies = {key for exp,key in final_graph_pairs if exp == graph_experiment}
    graph_label = 'Binance matched / cap weights' if graph_experiment == 'binance_matched_cap' else 'CMC 30 / cap weights'
    graph_currency = 'USDT' if graph_experiment == 'binance_matched_cap' else 'USD'
    graph_bitcoin_prices = (binance_prices['BTCUSDT'] if graph_currency == 'USDT'
                           else baseline_prices['BTC-USD'])
    equity_period_frames = {}
    for period,dates in PERIOD_DATES.items():
        data = {}
        for key in STRATEGIES:
            if key not in final_graph_strategies: continue
            nav = results[graph_experiment,period,key,'net'].nav
            if not nav.index.equals(dates): raise ValueError('Strategy ledger dates do not match requested period.')
            data[key] = nav
        data['bitcoin_hold'] = bitcoin_hold_nav(dates, graph_bitcoin_prices)
        frame = pd.DataFrame(data,index=dates)
        initial = pd.DataFrame(INITIAL_CAPITAL,index=[dates[0]-pd.Timedelta(days=1)],columns=frame.columns)
        equity_period_frames[period] = pd.concat([initial,frame])
        draw_equity_comparison(equity_period_frames[period],f'{graph_label} / {period}: strategies and Bitcoin',
            f'{graph_experiment}_{period}',
            'Each period starts with the same capital. Bitcoin is bought at the preceding daily close.')

    # compound existing net ledgers without injecting fresh capital.
    # Preserve backtest liquidation and validation restart; this is NOT a new uninterrupted simulation.
    if {'Backtest','Validation'}.issubset(equity_period_frames):
        bt_dates,va_dates = PERIOD_DATES['Backtest'],PERIOD_DATES['Validation']
        if va_dates[0] != bt_dates[-1]+pd.Timedelta(days=1):
            raise ValueError('Combined graph requires adjacent backtest and validation dates.')
        combined_dates = bt_dates.append(va_dates)
        data = {}
        for key in STRATEGIES:
            if key not in final_graph_strategies: continue
            back = results[graph_experiment,'Backtest',key,'net'].nav
            valid = results[graph_experiment,'Validation',key,'net'].nav
            data[key] = pd.concat([back,valid*(back.iloc[-1]/INITIAL_CAPITAL)])
        # One continuous BTC hold: do not liquidate/rebuy at the validation boundary.
        data['bitcoin_hold'] = bitcoin_hold_nav(combined_dates, graph_bitcoin_prices)
        combined_equity = pd.DataFrame(data,index=combined_dates)
        initial = pd.DataFrame(INITIAL_CAPITAL,index=[bt_dates[0]-pd.Timedelta(days=1)],columns=combined_equity.columns)
        combined_equity = pd.concat([initial,combined_equity])
        draw_equity_comparison(combined_equity,f'{graph_label}: combined backtest and validation',
            f'{graph_experiment}_Combined',
            'Strategies: linked net periods with the existing boundary restart. Bitcoin: one continuous hold.',
            boundary=va_dates[0])
    else:
        print('Combined graph requires both periods; run with RUN_SCOPE = both.')

    equity_by_universe[graph_experiment] = equity_period_frames
spec['limitations']=['USDT pairs only; other quotes audited','CMC mappings provisional and historical cohort restricted',
 'Zero caps unavailable; exact Sunday caps only','Missing held quotes stop account; partial ledgers unranked',
 'Spot shorts hypothetical; no borrow/funding/margin data','Original USD and Binance USDT accounts not currency identical',
 'Maximum price retains original rolling-close proxy','Dates previously explored; no prospective claim']
(OUTPUT_DIR/'manifest.json').write_text(json.dumps(spec,indent=2),encoding='utf8')
print('Saved comparison, failures, ledgers, constituent weights, coverage, charts and manifest to', OUTPUT_DIR.relative_to(NOTEBOOK_DIR).as_posix()+'/')

Saved comparison, failures, ledgers, constituent weights, coverage, charts and manifest to output/Common_Risk_Factors_Liu_et_al/


## 9. Compare backtest and validation Sharpe ratios

Rank eligible experiment/strategy pairs once by validation Sharpe and once by backtest Sharpe;
show both period scores for each pair on a shared axis. Only CMC 30 and Binance matched-cap
pairs complete in both periods with finite net Sharpe qualify. There can be at most 18 entries,
and fewer if some accounts fail or have undefined Sharpe. Missing results are not zero-filled.
Bitcoin remains an equity benchmark, not an additional ranked strategy here. Rankings are descriptive.


In [10]:
# compare up to 20 experiment/strategy combinations, ranked separately by period.
# Include CMC 30 and matched cap-weighted Binance strategies, requiring completion in BOTH periods.
# Uses NET-of-cost Sharpe; failed/missing results are never zero-filled.
# This cell can run after the comparison cells or directly from the latest saved results.
from pathlib import Path
from html import escape
import numpy as np
import pandas as pd
try:
    from IPython.display import display, HTML
except ImportError:
    def display(x): print(x.to_string() if hasattr(x, 'to_string') else x)
    def HTML(x): return x

if 'OUTPUT_DIR' in globals() and (Path(OUTPUT_DIR) / 'net_rankings.csv').exists():
    sharpe_output_dir = Path(OUTPUT_DIR)
else:
    sharpe_output_dir = Path('output') / 'Common_Risk_Factors_Liu_et_al'
    if not (sharpe_output_dir / 'net_rankings.csv').is_file():
        raise FileNotFoundError('Run the strategy comparison and export cells first.')

import json
sharpe_run_manifest = json.loads((sharpe_output_dir / 'manifest.json').read_text(encoding='utf8'))
sharpe_requested_periods = {
    'Backtest':[globals().get('BACKTEST_START','2021-01-02'), globals().get('BACKTEST_END','2024-12-31')],
    'Validation':[globals().get('VALIDATION_START','2025-01-01'), globals().get('VALIDATION_END','2026-09-17')],
}
if sharpe_run_manifest.get('periods') != sharpe_requested_periods:
    raise ValueError('Saved rankings use different dates. Update the source snapshots and rerun the comparison/export cells for the fixed dates before plotting.')

sharpe_data = (net_comparison.copy() if 'net_comparison' in globals()
               else pd.read_csv(sharpe_output_dir / 'net_rankings.csv'))
sharpe_data = sharpe_data.loc[
    sharpe_data['Experiment'].isin(['cmc_30_cap', 'binance_matched_cap'])
    & sharpe_data['Costs'].eq('net') & sharpe_data['Period'].isin(['Backtest', 'Validation'])
].copy()
sharpe_keys = ['Experiment', 'Strategy']
if sharpe_data.duplicated(sharpe_keys + ['Period']).any():
    raise ValueError('Expected one net result per experiment, strategy and period.')
sharpe_data['Sharpe'] = pd.to_numeric(sharpe_data['Sharpe'], errors='coerce')
sharpe_data['available_sharpe'] = sharpe_data['Sharpe'].where(
    sharpe_data['Status'].eq('complete') & np.isfinite(sharpe_data['Sharpe'])
)
sharpe_wide = sharpe_data.pivot(index=sharpe_keys, columns='Period', values='available_sharpe')
sharpe_wide = sharpe_wide.reindex(columns=['Backtest', 'Validation'])
sharpe_status = sharpe_data.pivot(index=sharpe_keys, columns='Period', values='Status')
sharpe_status = sharpe_status.reindex(columns=['Backtest', 'Validation'])

sharpe_eligible = (sharpe_status.eq('complete').all(axis=1)
                   & np.isfinite(sharpe_wide).all(axis=1))
sharpe_wide = sharpe_wide.loc[sharpe_eligible]
sharpe_status = sharpe_status.loc[sharpe_eligible]

sharpe_strategy_names = {
    'size': 'Size', 'price': 'Price', 'max_price': 'Maximum close (7-day proxy)',
    'mom_1w': 'Momentum 1 week', 'mom_2w': 'Momentum 2 weeks',
    'mom_3w': 'Momentum 3 weeks', 'mom_4w': 'Momentum 4 weeks',
    'dollar_volume': 'Volume level', 'volume_volatility': 'Volume volatility',
}
sharpe_experiment_names = {
    'cmc_30_cap': 'CMC 30 / cap weights',
    'binance_matched_cap': 'Binance matched / cap weights',
}
sharpe_colors = {'Backtest': '#2563eb', 'Validation': '#d97706'}
# one shared axis for both charts, including zero and negative Sharpe values.
sharpe_finite = sharpe_wide.to_numpy().ravel()
sharpe_finite = sharpe_finite[np.isfinite(sharpe_finite)]
sharpe_low = min(0.0, float(sharpe_finite.min())) if len(sharpe_finite) else -1.0
sharpe_high = max(0.0, float(sharpe_finite.max())) if len(sharpe_finite) else 1.0
sharpe_padding = max(sharpe_high - sharpe_low, 1.0) * .08
sharpe_low -= sharpe_padding
sharpe_high += sharpe_padding

def draw_sharpe_ranking(ranking_period, limit=20):
    ranked = (sharpe_wide.reset_index().dropna(subset=[ranking_period])
              .sort_values([ranking_period, 'Experiment', 'Strategy'],
                           ascending=[False, True, True], kind='stable')
              .head(limit).set_index(sharpe_keys))
    available = int(sharpe_wide[ranking_period].notna().sum())
    ranked.insert(0, 'Rank', range(1, len(ranked) + 1))
    for period in ['Backtest', 'Validation']:
        ranked[period + ' status'] = sharpe_status[period].reindex(ranked.index)
    title = f'Top {len(ranked)} by {ranking_period.lower()} Sharpe'
    left, right, top, row_height = 505, 1120, 142, 48
    bottom = top + max(len(ranked), 1) * row_height
    width, height = 1400, bottom + 80
    def x(value):
        return left + (value - sharpe_low) / (sharpe_high - sharpe_low) * (right - left)
    svg = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {width} {height}" '
           f'role="img" aria-label="{escape(title)}" style="font-family:Arial,sans-serif;max-width:100%;height:auto">',
           f'<rect width="{width}" height="{height}" fill="white"/>',
           f'<text x="24" y="32" font-size="23" font-weight="bold" fill="#0f172a">{escape(title)}</text>',
           f'<text x="24" y="57" font-size="13" fill="#475569">Net of costs | {len(ranked)} shown of {available} eligible strategies | CMC 30 + matched cap-weighted Binance; complete in both periods</text>',
           '<text x="24" y="80" font-size="13" fill="#475569">Ranked descending in the named period; the other period is shown for comparison. These are descriptive rankings.</text>']
    for period, legend_x in [('Backtest', 25), ('Validation', 170)]:
        svg.extend([f'<rect x="{legend_x}" y="99" width="15" height="13" fill="{sharpe_colors[period]}"/>',
                    f'<text x="{legend_x + 22}" y="110" font-size="13">{period}</text>'])
    svg.extend(['<text x="1160" y="126" font-size="12" fill="#2563eb">Backtest</text>',
                '<text x="1270" y="126" font-size="12" fill="#d97706">Validation</text>'])
    for tick in np.linspace(sharpe_low, sharpe_high, 7):
        svg.extend([f'<path d="M{x(tick):.2f} {top} V{bottom}" stroke="#e2e8f0"/>',
                    f'<text x="{x(tick):.2f}" y="{bottom+24}" text-anchor="middle" font-size="12" fill="#475569">{tick:.2f}</text>'])
    svg.append(f'<path d="M{x(0):.2f} {top} V{bottom}" stroke="#64748b" stroke-width="1.5"/>')
    for i, ((experiment, strategy), row) in enumerate(ranked.iterrows()):
        y = top + i * row_height
        label = (f'{int(row["Rank"]):02}. {sharpe_strategy_names.get(strategy, strategy)}'
                 f' | {sharpe_experiment_names.get(experiment, experiment)}')
        svg.append(f'<text x="24" y="{y+26}" font-size="12" fill="#0f172a">{escape(label)}</text>')
        svg.append(f'<path d="M24 {y+46} H1370" stroke="#f1f5f9"/>')
        for period, offset, number_x in [('Backtest', 5, 1160), ('Validation', 23, 1270)]:
            value = row[period]
            if pd.notna(value):
                bar_left, bar_width = min(x(0), x(value)), abs(x(value) - x(0))
                svg.append(f'<rect x="{bar_left:.2f}" y="{y+offset}" width="{bar_width:.2f}" height="14" fill="{sharpe_colors[period]}"><title>{period}: {value:.4f}</title></rect>')
                text_value = f'{value:+.3f}'
            else:
                state = row[period + ' status']
                text_value = {'needs_data_review':'N/A: review', 'execution_unavailable':'N/A: exit',
                              'insolvent':'Insolvent','failed':'N/A: legacy'}.get(state, 'N/A')
            svg.append(f'<text x="{number_x}" y="{y+27}" font-size="12" fill="{sharpe_colors[period]}">{text_value}</text>')
    if ranked.empty:
        svg.append(f'<text x="24" y="{top+25}" font-size="14">No CMC 30 or matched cap-weighted Binance strategies have completed both periods with finite Sharpe scores.</text>')
    svg.append(f'<text x="{(left+right)/2}" y="{bottom+48}" text-anchor="middle" font-size="13">Sharpe (252) ratio</text></svg>')
    chart = ''.join(svg)
    stem = f'top20_sharpe_ranked_by_{ranking_period.lower()}'
    (sharpe_output_dir / (stem + '.svg')).write_text(chart, encoding='utf8')
    ranked.to_csv(sharpe_output_dir / (stem + '.csv'))
    display(HTML(chart))
    if available < limit:
        print(f'{ranking_period}: only {available} CMC 30 or matched cap-weighted Binance strategies complete in both periods with finite Sharpe; showing all {available}.')
    return ranked

top20_by_validation = draw_sharpe_ranking('Validation')
top20_by_backtest = draw_sharpe_ranking('Backtest')
print('Charts and ranked tables saved to: output/Common_Risk_Factors_Liu_et_al/')


Validation: only 18 CMC 30 or matched cap-weighted Binance strategies complete in both periods with finite Sharpe; showing all 18.


Backtest: only 18 CMC 30 or matched cap-weighted Binance strategies complete in both periods with finite Sharpe; showing all 18.
Charts and ranked tables saved to: output/Common_Risk_Factors_Liu_et_al/


## 10. Interpretation, limitations and audit trail

This notebook asks whether the paper-inspired rankings persist across a fixed 30-coin CMC panel
and a broader cap-matched Binance panel when both use cap weighting. Binance matched-cap is
closer to the paper's broad-universe portfolio construction; CMC 30 more closely matches its price
vendor. Neither reproduces the paper's 2014-2018 sample, weekly calendar, full historical universe,
all variable definitions or weekly winsorization. Maximum price remains a rolling-close proxy.

The nine signals are separate accounts, not a combined multi-factor allocation. The paper's
three-factor regression is not estimated. Dollar neutrality at rebalance is not beta neutrality.
Market-cap eligibility is distinct from a liquidity screen: neither experiment applies our removed
30-day median-turnover filter. More complete data is not evidence of a profitable strategy.

Read `failures.csv` and partial ledgers before interpreting missing rankings. `needs_data_review`
means an observation cannot be accepted, `execution_unavailable` means a required trade cannot
be substantiated, and `insolvent` means modeled capital was exhausted. Only `complete` paths
receive full-period performance metrics. Event repairs preserve observed prices, costs and history;
no coin or failed date is discarded merely to produce completion.

The generated status summary establishes which experiments completed in this run; completion is not assumed for future input revisions.

Borrow availability, funding/borrow costs and margin liquidations remain unmodeled. Source vintages
and historical CMC cohort coverage have limitations; the fixed 30-coin selection can introduce
selection/survivorship bias. Both periods have already been examined, so validation is not a fresh
prospective test. Review the generated results rather than importing a winner from another universe.
